# Olist Delivery-Delay EDA
### Evidence for a Real-Time Supply-Chain Delay Prediction System (Apache Kafka + ML)

**Purpose.** This notebook backs every design decision of the project with data: what "late" means, which columns carry signal, why we score an order at **three stages** instead of one, which rule-based checks are worth running in the stream, and what customer reviews can (and cannot) tell us.

**Inputs (two files, same folder as the notebook)**
* `olist_master_orders.csv`: one row per order, built by joining the 9 Olist files (51 columns). The label is the column **`late`** (1 = delivered after the promised date, 0 = on time, blank = not delivered).
* `olist_order_reviews_dataset.csv` (or its `.zip`): customer reviews, Portuguese text. Used only for *post-delivery* analysis, never as a model input.

**How to read it.** Every section opens with the question it answers and every chart ends with a one-line takeaway. All headline numbers are stored and re-checked in the last section (*Reproduction check*).

| Section | Question |
|---|---|
| 0 Setup | Load the data |
| 1 Dataset & label | What is "late", and how big is the problem? |
| 2 Data quality | Can we trust the joins and timestamps? |
| 3 Purchase-time signals | What do we know at checkout that relates to lateness? |
| 4 Payments & approval | Does payment type / slow approval matter? |
| 5 Stages | **Why three scoring stages?** |
| 6 Time & drift | Why a chronological split and no month feature |
| 7 Sellers | Are some sellers consistently late? |
| 8 Label quality | Multi-seller orders |
| 9 Correlations | Which signals are strongest, which overlap? |
| 10 Reviews | What do customers say? |
| 11 Rules | Which stream rules R1 to R9 earn their place? |
| 12 Conclusions | Reproduction check and final design |

## 0. Setup

In [ ]:
import warnings, glob, os, re, unicodedata
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import Patch
import seaborn as sns
from scipy.stats import spearmanr, rankdata
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12,
                     "axes.labelsize": 10, "figure.figsize": (10, 4.5),
                     "axes.spines.top": False, "axes.spines.right": False})
C_LATE, C_OK, C_GREY, C_ACC, C_GREEN = "#d1495b", "#2e86ab", "#8d99ae", "#edae49", "#3d9970"

res = {}   # every headline number is stored here and verified in section 12


def say(text):
    display(Markdown("> **Takeaway:** " + text))


def find_file(patterns, dirs=(".", "data", "/mnt/user-data/uploads")):
    for d in dirs:
        for p in patterns:
            hits = sorted(glob.glob(os.path.join(d, p)))
            if hits:
                return hits[0]
    raise FileNotFoundError(f"Could not find any of {patterns}. Put the file next to the notebook.")


def auc_uni(x, y):
    """AUC of ONE column vs the late flag (0.5 = no signal, 1.0 = perfect). Rank based, no model."""
    x = pd.Series(np.asarray(x, dtype=float)).reset_index(drop=True)
    y = np.asarray(y)
    ok = x.notna().values
    if ok.sum() < 1000:
        return np.nan
    r = rankdata(x[ok]); yy = y[ok]
    n1 = yy.sum(); n0 = len(yy) - n1
    return (r[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def rate_by(d, key, minn=500, sort=False):
    """Late rate (%) and order count per group; hides groups smaller than minn."""
    g = d.groupby(key, observed=True).late.agg(n="size", late_pct="mean")
    g["late_pct"] = g["late_pct"] * 100
    g = g[g.n >= minn]
    return g.sort_values("late_pct") if sort else g


def bar_rate(ax, tbl, title, xlabel="", rot=0, color=C_LATE, base=None, show_n=True, fs=7.5):
    x = [str(i) for i in tbl.index]
    bars = ax.bar(x, tbl.late_pct, color=color, edgecolor="white")
    for b, v, n in zip(bars, tbl.late_pct, tbl.n):
        lab = f"{v:.1f}%\nn={n:,}" if show_n else f"{v:.1f}"
        ax.annotate(lab, (b.get_x() + b.get_width() / 2, v), ha="center", va="bottom", fontsize=fs)
    if base is not None:
        ax.axhline(base, ls="--", c=C_GREY, lw=1)
        ax.text(len(x) - 0.5, base, f" overall {base:.1f}%", va="bottom", ha="right", fontsize=8, color=C_GREY)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Late rate (%)")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    ax.set_ylim(0, max(tbl.late_pct.max() * 1.3, 1))
    ax.tick_params(axis="x", rotation=rot)

In [ ]:
MASTER = find_file(["olist_master_orders.csv"])
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date", "ship_limit"]
df = pd.read_csv(MASTER, parse_dates=date_cols)
df["delivered"] = df["order_delivered_customer_date"].notna()
D = df[df.delivered & df.late.notna()].copy()          # delivered orders: the only ones with a label
BASE = D.late.mean() * 100
res.update(rows=len(df), cols=df.shape[1], delivered=len(D), late_rate=BASE)
print(f"master file : {MASTER}")
print(f"all orders  : {len(df):,} rows x {df.shape[1]} columns")
print(f"delivered   : {len(D):,}  (these carry the label)")
print(f"late rate   : {BASE:.2f}%   (late = delivered date AFTER promised date)")
display(df[["order_id", "order_status", "order_purchase_timestamp", "order_estimated_delivery_date",
            "order_delivered_customer_date", "dist_km", "freight", "pay_type", "late", "days_late"]].head(5))

In [ ]:
# Data dictionary: every one of the 51 columns, its role and the moment it becomes known
DICT = {
 "order_id": ("ID", "key", "order key"), "customer_id": ("ID", "key", "per-order customer key"),
 "order_status": ("Outcome", "final", "final status; used only to filter and for dashboards, never an input"),
 "order_purchase_timestamp": ("Timestamp", "purchase", "event time of 'order created'; source of hour, weekday"),
 "order_approved_at": ("Timestamp", "approval", "event time of 'approved'"),
 "order_delivered_carrier_date": ("Timestamp", "handoff", "event time of 'handed to carrier'"),
 "order_delivered_customer_date": ("Outcome", "delivery", "delivery time; defines the label"),
 "order_estimated_delivery_date": ("Timestamp", "purchase", "promised delivery date"),
 "n_items": ("V1 feature", "purchase", "items in the order"), "n_sellers": ("Filter", "purchase", ">1 seller: label unreliable"),
 "n_products": ("Excluded", "purchase", "duplicates n_items"), "price": ("V1 feature", "purchase", "sum of item prices"),
 "freight": ("V1 feature", "purchase", "sum of freight"), "weight_g": ("V1 feature", "purchase", "total weight"),
 "vol_cm3": ("V1 feature", "purchase", "total volume"), "max_len": ("Excluded", "purchase", "longest side, no signal"),
 "photos": ("Excluded", "purchase", "marketing field"), "name_len": ("Excluded", "purchase", "marketing field"),
 "desc_len": ("Excluded", "purchase", "marketing field"), "category": ("V1 feature", "purchase", "Portuguese category (first item)"),
 "seller_id": ("ID", "purchase", "key for seller history"), "seller_state": ("V1 feature", "purchase", "seller state"),
 "sl_lat": ("Helper", "purchase", "seller latitude (zip centroid)"), "sl_lng": ("Helper", "purchase", "seller longitude"),
 "ship_limit": ("Helper", "purchase", "seller's shipping deadline"), "customer_unique_id": ("ID", "purchase", "the real person"),
 "customer_zip_code_prefix": ("Helper", "purchase", "used to place the customer"),
 "customer_state": ("V1 feature", "purchase", "customer state"), "pay_n": ("Excluded", "purchase", "number of payment rows"),
 "pay_value": ("Excluded", "purchase", "duplicates price + freight"), "installments": ("V1 feature", "purchase", "max installments"),
 "pay_type": ("V1 feature", "purchase", "main payment type"), "cu_lat": ("Helper", "purchase", "customer latitude"),
 "cu_lng": ("Helper", "purchase", "customer longitude"), "dist_km": ("V1 feature", "purchase", "seller-to-customer distance"),
 "same_state": ("V1 feature", "purchase", "seller and customer in same state"),
 "promise_d": ("V1 feature", "purchase", "promised window = estimated date - purchase (days)"),
 "appr_h": ("Stage-2 feature", "approval", "hours from purchase to approval"),
 "limit_slack_d": ("Excluded", "purchase", "no signal"), "handling_d": ("Stage-3 feature", "handoff", "days from purchase to carrier"),
 "seller_overrun_d": ("Stage-3 feature", "handoff", "carrier date minus shipping limit (days)"),
 "transit_d": ("Outcome", "delivery", "carrier to customer (days)"),
 "post_appr_d": ("Stage-3 feature", "handoff", "days from approval to carrier"),
 "days_late": ("Outcome", "delivery", "delivered minus promised (days)"), "delivered": ("Outcome", "delivery", "has a delivery date"),
 "late": ("LABEL", "delivery", "1 = delivered after promised date"),
 "p_month": ("Excluded", "purchase", "month: regime shift, not seasonality"),
 "p_dow": ("V1 feature", "purchase", "weekday"), "p_hour": ("V1 feature", "purchase", "hour"),
 "freight_ratio": ("Excluded", "purchase", "no signal"), "category_en": ("Dashboard label", "purchase", "English category name"),
}
dd_ = pd.DataFrame({"column": df.columns})
dd_["group"] = dd_.column.map(lambda c: DICT.get(c, ("?", "", ""))[0])
dd_["known_at"] = dd_.column.map(lambda c: DICT.get(c, ("?", "", ""))[1])
dd_["note"] = dd_.column.map(lambda c: DICT.get(c, ("?", "", ""))[2])
dd_["dtype"] = [str(df[c].dtype) for c in df.columns]
dd_["%null"] = [df[c].isna().mean() * 100 for c in df.columns]
display(dd_.groupby("group").size().rename("columns").to_frame().T)
display(dd_)
res["n_v1_features"] = int((dd_.group == "V1 feature").sum())
say(f"All {len(dd_)} columns are accounted for. {res['n_v1_features']} are V1 purchase-time features; the rest are IDs, helpers, later-stage features, outcomes or excluded columns.")

## 1. Dataset and label
*Question: how big is the data, what exactly is "late", and how many orders can we learn from?*

In [ ]:
inv = pd.DataFrame([
    ("olist_orders_dataset", 99441, 8, "order lifecycle timestamps: the spine and the label source"),
    ("olist_order_items_dataset", 112650, 7, "price, freight, seller, shipping deadline (several rows per order)"),
    ("olist_products_dataset", 32951, 9, "weight, size, category"),
    ("olist_customers_dataset", 99441, 5, "customer state / zip"),
    ("olist_sellers_dataset", 3095, 4, "seller state / zip"),
    ("olist_geolocation_dataset", 1000163, 5, "zip-prefix coordinates, collapsed to one centroid per prefix"),
    ("olist_order_payments_dataset", 103886, 5, "payment type, installments (several rows per order)"),
    ("olist_order_reviews_dataset", 99224, 7, "post-delivery only; not a model input"),
    ("product_category_name_translation", 71, 2, "English category names for dashboards"),
], columns=["file", "rows", "columns", "role in the project"])
display(inv)
res["orig_columns"] = int(inv["columns"].sum())
say(f"The nine source files hold {res['orig_columns']} columns. We join them into one order-level master table with {df.shape[1]} columns (one row per order), aggregating items and payments first so rows never multiply.")

In [ ]:
# Orders over time (adapted from the reference notebook): is the platform growing, and when is the data dense?
m_all = df.groupby("p_month").size()
fig, ax = plt.subplots(figsize=(13, 4.2))
cols = [C_GREY if n < 500 else C_OK for n in m_all.values]
ax.bar(m_all.index, m_all.values, color=cols)
ax.set_title("Orders per purchase month (grey = sparse months, not used for rates)"); ax.set_ylabel("orders")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()
jan_aug_17 = df[(df.p_month >= "2017-01") & (df.p_month <= "2017-08")].shape[0]
jan_aug_18 = df[(df.p_month >= "2018-01") & (df.p_month <= "2018-08")].shape[0]
res["growth_jan_aug"] = (jan_aug_18 / jan_aug_17 - 1) * 100
say(f"Order volume grows strongly: Jan-Aug 2018 has {res['growth_jan_aug']:.0f}% more orders than Jan-Aug 2017. 2016 is sparse, so rates are only trusted from 2017-01 onward.")

In [ ]:
# When do customers buy? (weekday / hour: both are model features)
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
dow = df.p_dow.value_counts().sort_index()
axs[0].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], dow.values, color=C_OK)
axs[0].set_title("Orders by weekday")
hr = df.p_hour.value_counts().sort_index()
axs[1].bar(hr.index, hr.values, color=C_OK); axs[1].set_title("Orders by hour of day"); axs[1].set_xlabel("hour")
plt.tight_layout(); plt.show()
peak_h = int(hr.idxmax())
say(f"Buying peaks around {peak_h}:00 and is lowest on weekends. Both are known at purchase, so they are cheap features, though (section 3) they carry little signal about lateness.")

In [ ]:
# The label: delivered date vs promised date, compared at DATE level
ts_late = D.order_delivered_customer_date > D.order_estimated_delivery_date     # naive full-timestamp compare
date_late = D.late == 1                                                          # our label (date compare)
res.update(late_n=int(date_late.sum()), ontime_n=int((D.late == 0).sum()), undelivered_n=int((~df.delivered).sum()),
           ts_late_rate=ts_late.mean() * 100, label_differ=int((ts_late != date_late).sum()))
print(pd.Series({"late (1)": res["late_n"], "on time (0)": res["ontime_n"], "not delivered (blank)": res["undelivered_n"]}).map("{:,}".format))
print(f"\nLate rate with our date-level rule : {BASE:.2f}%")
print(f"Late rate with naive timestamp rule: {res['ts_late_rate']:.2f}%   ({res['label_differ']:,} orders differ)")
ex = D[ts_late & ~date_late][["order_estimated_delivery_date", "order_delivered_customer_date", "late"]].head(4)
display(ex)
say("The promised date is stored at 00:00, so any delivery on the promised day looks 'late' if full timestamps are compared. Delivering on the promised day is on time, so we compare dates.")

In [ ]:
# How late are late orders? (severity matters more than the flag)
dl = (D.order_delivered_customer_date.dt.normalize() - D.order_estimated_delivery_date.dt.normalize()).dt.days
res["median_days_late"] = float(dl[D.late == 1].median())
sev = pd.cut(dl, [-1000, 0, 3, 7, 14, 1000], labels=["on time / early", "1-3 days late", "4-7 days late", "8-14 days late", ">14 days late"])
sev_tbl = sev.value_counts().reindex(sev.cat.categories).to_frame("orders")
sev_tbl["share_%"] = sev_tbl.orders / sev_tbl.orders.sum() * 100
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].hist(dl.clip(-40, 60), bins=100, color=C_OK)
axs[0].axvline(0, c=C_LATE); axs[0].axvline(3, c=C_ACC, ls="--")
axs[0].set_title("Delivery date minus promised date (days, clipped)"); axs[0].set_xlabel("days (0 = promised day)")
axs[1].barh(sev_tbl.index[::-1].astype(str), sev_tbl.orders[::-1], color=[C_LATE, C_LATE, C_ACC, C_ACC, C_OK])
axs[1].set_title("Severity buckets"); axs[1].set_xscale("log")
plt.tight_layout(); plt.show()
display(sev_tbl)
say(f"Most orders arrive early. Late orders are typically {res['median_days_late']:.0f} days past the promise, so a 'severely late' flag (3+ days) is a meaningful second signal (see section 10).")

In [ ]:
# Order status and the 'stuck' orders that have no label
status = df.order_status.value_counts()
nd = df[~df.delivered]
end_of_data = df.order_delivered_customer_date.max()
stuck = nd[nd.order_estimated_delivery_date < end_of_data]
res.update(stuck_n=len(stuck), status_delivered_nodate=int(((df.order_status == "delivered") & ~df.delivered).sum()),
           date_notdelivered=int(((df.order_status != "delivered") & df.delivered).sum()))
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].barh(status.index[::-1], status.values[::-1], color=C_OK); axs[0].set_xscale("log"); axs[0].set_title("Orders by final status (log scale)")
sb = nd.order_status.value_counts()
axs[1].bar(sb.index, sb.values, color=C_LATE); axs[1].set_title(f"Orders with no delivery date, by status (n={len(nd):,})"); axs[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
print(f"Orders with no delivery date            : {len(nd):,}")
print(f"  ...already past their promised date   : {len(stuck):,}   <- rule R4 'already late / stuck'")
print(f"Status 'delivered' but no delivery date : {res['status_delivered_nodate']}")
print(f"Delivery date but status not 'delivered': {res['date_notdelivered']}")
say(f"{len(stuck):,} orders never received a delivery date yet are past their promise. The model can only learn from delivered orders, so these are caught by a stream rule (R4), and we state this limitation in the report.")

## 2. Data quality
*Question: can we trust the joins, and are there timestamp problems the Kafka replay must handle?*

In [ ]:
has_items = df[df.n_items.notna()]
res.update(multi_item=(has_items.n_items > 1).mean() * 100, multi_seller_orders=int((df.n_sellers > 1).sum()),
           multi_pay=(df[df.pay_n.notna()].pay_n > 1).mean() * 100, no_items=int(df.n_items.isna().sum()))
q = pd.DataFrame({
    "orders affected": [int((has_items.n_items > 1).sum()), int((df.n_sellers > 1).sum()), int((df.pay_n > 1).sum()), int(df.n_items.isna().sum())],
    "share of orders (%)": [res["multi_item"], (df.n_sellers > 1).sum() / len(has_items) * 100, res["multi_pay"], df.n_items.isna().mean() * 100]},
    index=["more than one item", "more than one seller", "more than one payment row", "no item rows at all"])
display(q)
say("Items and payments are one-to-many with orders. We aggregate them to one row per order before joining, otherwise orders (and the label) would be duplicated. Orders without items are the never-delivered ones.")

In [ ]:
chk_cols = ["category", "photos", "name_len", "desc_len", "dist_km", "weight_g", "vol_cm3", "pay_type", "installments",
            "order_approved_at", "order_delivered_carrier_date", "seller_overrun_d", "same_state"]
nul = pd.DataFrame({"delivered orders %": [D[c].isna().mean() * 100 for c in chk_cols],
                    "all orders %": [df[c].isna().mean() * 100 for c in chk_cols]}, index=chk_cols)
ax = nul.sort_values("delivered orders %").plot.barh(figsize=(9, 5), color=[C_OK, C_GREY])
ax.set_title("Missing values in key columns"); ax.set_xlabel("% missing")
plt.tight_layout(); plt.show()
res.update(null_category=D.category.isna().mean() * 100, null_dist=D.dist_km.isna().mean() * 100)
say(f"Missing data is small: category {res['null_category']:.1f}%, distance {res['null_dist']:.1f}% of delivered orders. Never-delivered orders explain the blanks in the timestamp columns.")

In [ ]:
Dh = D.copy()
anom = pd.Series({
    "approved before purchase": (Dh.order_approved_at < Dh.order_purchase_timestamp).sum(),
    "carrier handoff before purchase": (Dh.order_delivered_carrier_date < Dh.order_purchase_timestamp).sum(),
    "carrier handoff before approval": (Dh.order_delivered_carrier_date < Dh.order_approved_at).sum(),
    "delivered before carrier handoff": (Dh.order_delivered_customer_date < Dh.order_delivered_carrier_date).sum(),
    "estimated date before purchase": (Dh.order_estimated_delivery_date < Dh.order_purchase_timestamp).sum(),
    "approval timestamp missing": Dh.order_approved_at.isna().sum(),
    "carrier timestamp missing": Dh.order_delivered_carrier_date.isna().sum(),
}).astype(int)
res.update(carrier_before_appr=int(anom["carrier handoff before approval"]), carrier_before_purchase=int(anom["carrier handoff before purchase"]),
           deliv_before_carrier=int(anom["delivered before carrier handoff"]))
display(anom.to_frame("delivered orders"))
say(f"In {res['carrier_before_appr']:,} orders (about {res['carrier_before_appr']/len(D)*100:.1f}%) the handoff is recorded before approval. When we replay events in time order, a scorer must tolerate an event arriving 'out of sequence', which is also a good event-time demo.")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].hist(D.dist_km.dropna(), bins=60, color=C_OK); axs[0].set_title("Seller-to-customer distance (km)"); axs[0].set_xlabel("km")
ss = D.same_state.value_counts(normalize=True).rename({1.0: "same state", 0.0: "different state"}) * 100
axs[1].bar(ss.index, ss.values, color=[C_OK, C_ACC]); axs[1].set_title("Share of orders by route type (%)")
plt.tight_layout(); plt.show()
res["median_dist"] = float(D.dist_km.median()); res["cross_state_share"] = float(ss.get("different state", np.nan))
say(f"Median route is {res['median_dist']:.0f} km and {res['cross_state_share']:.0f}% of orders cross state lines. Distance is computed from zip-prefix centroids (Brazil bounding box applied), so it is approximate.")

## 3. Purchase-time signals
*Question: what is known at checkout (the first scoring point) and how does each piece relate to lateness?*
Every chart below uses **delivered orders only**, hides groups with too few orders, and draws the overall late rate as a dashed line.

In [ ]:
st = D.customer_state.value_counts()
fig, axs = plt.subplots(2, 1, figsize=(13, 8.5))
top = st.head(15)
axs[0].bar(top.index, top.values / len(D) * 100, color=C_OK)
axs[0].set_title("Share of delivered orders by customer state (top 15)"); axs[0].set_ylabel("% of orders")
tbl_state = rate_by(D, "customer_state", 500, sort=True)
bar_rate(axs[1], tbl_state, "Late rate by customer state (states with 500+ orders)", show_n=False, base=BASE)
plt.tight_layout(); plt.show()
for s_ in ["PR", "SP", "CE", "MA"]:
    res[f"rate_{s_}"] = float(tbl_state.late_pct.get(s_, np.nan))
res["share_SP"] = float(st.get("SP", 0) / len(D) * 100)
display(pd.concat([tbl_state.head(4), tbl_state.tail(4)]).rename(columns={"n": "orders", "late_pct": "late %"}))
say(f"Where the customer lives matters a lot: late rate runs from {tbl_state.late_pct.min():.1f}% to {tbl_state.late_pct.max():.1f}% across states, while São Paulo alone is {res['share_SP']:.0f}% of orders (a partition-skew point for Kafka).")

In [ ]:
D["dist_bin"] = pd.cut(D.dist_km, [0, 100, 300, 600, 1000, 1500, 5000])
t_dist = rate_by(D, "dist_bin", 500)
fig, ax = plt.subplots(figsize=(10, 4.2))
bar_rate(ax, t_dist, "Late rate by seller-to-customer distance", xlabel="km", base=BASE)
plt.tight_layout(); plt.show()
res["dist_first"] = float(t_dist.late_pct.iloc[0]); res["dist_last"] = float(t_dist.late_pct.iloc[-1])
res["auc_dist"] = auc_uni(D.dist_km, D.late)
say(f"Late rate climbs from {res['dist_first']:.1f}% under 100 km to {res['dist_last']:.1f}% beyond 1,500 km. Distance alone ranks orders only moderately (AUC {res['auc_dist']:.2f}), so expect a modest model, not a spectacular one.")

In [ ]:
g = D.groupby(["same_state", "dist_bin"], observed=True).late.agg(n="size", late_pct="mean")
g["late_pct"] *= 100
g.loc[g.n < 300, "late_pct"] = np.nan
piv = g.late_pct.unstack("same_state").rename(columns={0.0: "cross-state", 1.0: "same state"})
ax = piv.plot.bar(figsize=(10, 4.2), color=[C_LATE, C_OK], width=0.8)
ax.set_title("Late rate by distance, split by same-state vs cross-state (cells with <300 orders hidden)")
ax.set_ylabel("Late rate (%)"); ax.set_xlabel("distance bin (km)"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
t_same = rate_by(D, "same_state", 500)
res["same_rate"] = float(t_same.late_pct.loc[1.0]); res["cross_rate"] = float(t_same.late_pct.loc[0.0])
display(t_same.rename(index={0.0: "cross-state", 1.0: "same state"}))
say(f"Cross-state routes are late {res['cross_rate']:.1f}% of the time vs {res['same_rate']:.1f}% in-state. Inside one state distance barely matters; it matters on long cross-state routes, which is where geolocation adds value beyond a simple state flag.")

In [ ]:
tbl_sstate = rate_by(D, "seller_state", 500, sort=True)
fig, ax = plt.subplots(figsize=(10, 4.2))
bar_rate(ax, tbl_sstate, "Late rate by SELLER state (500+ orders)", base=BASE, show_n=True, fs=7)
plt.tight_layout(); plt.show()
say(f"Seller state is a much weaker signal ({tbl_sstate.late_pct.min():.1f}% to {tbl_sstate.late_pct.max():.1f}%) than customer state. Most late orders are explained at the delivery end of the route.")

In [ ]:
sub = D[D.dist_km.between(600, 1500)]
t_ctrl = rate_by(sub, "customer_state", 400, sort=True)
fig, ax = plt.subplots(figsize=(11, 4.2))
bar_rate(ax, t_ctrl, "Customer state effect AFTER controlling distance (routes of 600-1,500 km only)", show_n=True, base=sub.late.mean() * 100, fs=7)
plt.tight_layout(); plt.show()
for s_ in ["SP", "MG", "RJ", "BA", "ES"]:
    res[f"ctrl_{s_}"] = float(t_ctrl.late_pct.get(s_, np.nan))
say(f"With distance held roughly equal, RJ ({res['ctrl_RJ']:.1f}%), BA ({res['ctrl_BA']:.1f}%) and ES ({res['ctrl_ES']:.1f}%) are still far later than SP ({res['ctrl_SP']:.1f}%) and MG ({res['ctrl_MG']:.1f}%). State captures route / carrier quality that distance cannot, so customer_state stays a feature even with distance.")

In [ ]:
D["freight_bin"] = pd.cut(D.freight, [0, 10, 15, 20, 30, 50, 10000])
t_fr = rate_by(D, "freight_bin", 500)
fm = D.groupby("customer_state").agg(n=("late", "size"), fr=("freight", "mean"))
fm = fm[fm.n >= 500].sort_values("fr")
fig, axs = plt.subplots(1, 2, figsize=(14, 4.4))
bar_rate(axs[0], t_fr, "Late rate by order freight (R$)", base=BASE)
axs[1].bar(fm.index, fm.fr, color=C_ACC); axs[1].set_title("Mean freight by customer state (R$)"); axs[1].tick_params(axis="x", rotation=90)
plt.tight_layout(); plt.show()
res["fr_first"] = float(t_fr.late_pct.iloc[0]); res["fr_last"] = float(t_fr.late_pct.iloc[-1]); res["auc_freight"] = auc_uni(D.freight, D.late)
say(f"Higher freight goes with higher lateness ({res['fr_first']:.1f}% to {res['fr_last']:.1f}%). Freight is mostly a price tag for distance and weight, so it overlaps with them (section 9).")

In [ ]:
# Delivery time by state (adapted from the reference notebook): promised vs actual days
D["actual_d"] = (D.order_delivered_customer_date - D.order_purchase_timestamp).dt.total_seconds() / 86400
sd = D.groupby("customer_state").agg(n=("late", "size"), actual=("actual_d", "mean"), promised=("promise_d", "mean"), late=("late", "mean"))
sd = sd[sd.n >= 500].sort_values("actual")
fig, ax = plt.subplots(figsize=(11, 7.5))
y_ = np.arange(len(sd))
ax.barh(y_ - 0.2, sd.promised, height=0.4, color=C_GREY, label="promised (days)")
ax.barh(y_ + 0.2, sd.actual, height=0.4, color=C_OK, label="actual (days)")
ax.set_yticks(y_); ax.set_yticklabels(sd.index); ax.legend(); ax.set_title("Mean promised vs actual delivery time by customer state")
plt.tight_layout(); plt.show()
res["state_gap_corr"] = float(np.corrcoef(sd.actual, sd.late * 100)[0, 1])
res["actual_mean"] = float(D.actual_d.mean()); res["promised_mean"] = float(D.promise_d.mean())
say(f"On average Olist promises {res['promised_mean']:.0f} days and delivers in {res['actual_mean']:.0f}: promises are padded. States with slow actual delivery are the late ones (correlation {res['state_gap_corr']:.2f}), so late orders are those that break through a generous buffer.")

In [ ]:
D["weight_bin"] = pd.cut(D.weight_g, [0, 300, 1000, 3000, 10000, 1e6])
D["vol_bin"] = pd.qcut(D.vol_cm3, 5, duplicates="drop")
fig, axs = plt.subplots(1, 2, figsize=(14, 4.4))
t_w = rate_by(D, "weight_bin", 500)
bar_rate(axs[0], t_w, "Late rate by total weight (g)", base=BASE)
bar_rate(axs[1], rate_by(D, "vol_bin", 500), "Late rate by total volume (cm3, quintiles)", base=BASE, fs=7)
plt.tight_layout(); plt.show()
res["w_first"] = float(t_w.late_pct.iloc[0]); res["w_last"] = float(t_w.late_pct.iloc[-1]); res["auc_weight"] = auc_uni(D.weight_g, D.late)
say(f"Heavy parcels are late a bit more often ({res['w_first']:.1f}% up to {res['w_last']:.1f}%), but weight is a weak signal on its own (AUC {res['auc_weight']:.2f}) and overlaps with freight.")

In [ ]:
t_cat = rate_by(D, "category_en", 800, sort=True)
sel = pd.concat([t_cat.head(8), t_cat.tail(8)])
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(sel.index, sel.late_pct, color=[C_OK] * 8 + [C_LATE] * 8)
ax.axvline(BASE, ls="--", c=C_GREY); ax.set_title("Lowest and highest late rate by product category (800+ orders)"); ax.set_xlabel("Late rate (%)")
plt.tight_layout(); plt.show()
res.update(cat_min=float(t_cat.late_pct.min()), cat_max=float(t_cat.late_pct.max()), cat_min_name=t_cat.late_pct.idxmin(), cat_max_name=t_cat.late_pct.idxmax())
say(f"Category barely separates late from on-time orders ({res['cat_min']:.1f}% for {res['cat_min_name']} to {res['cat_max']:.1f}% for {res['cat_max_name']}). It is kept as a cheap feature (top 20 + 'other') but is not a driver.")

In [ ]:
D["price_bin"] = pd.cut(D.price, [0, 50, 100, 200, 500, 1e5])
D["inst_bin"] = pd.cut(D.installments, [-1, 1, 3, 6, 10, 50])
D["items_bin"] = pd.cut(D.n_items, [0, 1, 2, 3, 50], labels=["1", "2", "3", "4+"])
fig, axs = plt.subplots(1, 3, figsize=(16, 4.2))
bar_rate(axs[0], rate_by(D, "price_bin", 500), "by order price (R$)", base=BASE, fs=7)
bar_rate(axs[1], rate_by(D, "inst_bin", 500), "by installments", base=BASE, fs=7)
bar_rate(axs[2], rate_by(D, "items_bin", 300), "by number of items", base=BASE, fs=7)
plt.tight_layout(); plt.show()
res["auc_price"] = auc_uni(D.price, D.late)
say("Price, installments and item count are nearly flat. They are kept only because they are free at checkout; none of them explains lateness by itself.")

In [ ]:
D["promise_bin"] = pd.cut(D.promise_d, [0, 10, 15, 20, 25, 30, 40, 200])
fig, axs = plt.subplots(1, 3, figsize=(17, 4.2))
t_pr = rate_by(D, "promise_bin", 300)
bar_rate(axs[0], t_pr, "by promised window (days)", base=BASE, fs=7)
t_h = rate_by(D, "p_hour", 500)
axs[1].plot(t_h.index, t_h.late_pct, marker="o", color=C_LATE); axs[1].axhline(BASE, ls="--", c=C_GREY)
axs[1].set_title("by purchase hour"); axs[1].set_xlabel("hour"); axs[1].set_ylabel("Late rate (%)")
t_d = rate_by(D, "p_dow", 500)
axs[2].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], t_d.late_pct, color=C_OK); axs[2].axhline(BASE, ls="--", c=C_GREY)
axs[2].set_title("by purchase weekday"); axs[2].set_ylabel("Late rate (%)")
plt.tight_layout(); plt.show()
res["promise_short"] = float(t_pr.late_pct.iloc[0]); res["promise_long"] = float(t_pr.late_pct.iloc[-1])
res["auc_promise"] = auc_uni(D.promise_d, D.late)
say(f"Tight promises break most: {res['promise_short']:.1f}% late when 10 days or less were promised vs {res['promise_long']:.1f}% when more than 40. The relationship is not a straight line (AUC {res['auc_promise']:.2f}), which suits tree models. Weekday and hour are close to flat.")

In [ ]:
# Where are the late orders? Customer locations on a lat/lng grid (no map tiles needed)
geo = D.dropna(subset=["cu_lat", "cu_lng"])
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
hb = axs[0].hexbin(geo.cu_lng, geo.cu_lat, gridsize=55, bins="log", cmap="Blues", mincnt=1)
axs[0].set_title("Where customers are (orders, log scale)"); fig.colorbar(hb, ax=axs[0], fraction=0.04)
hb2 = axs[1].hexbin(geo.cu_lng, geo.cu_lat, C=geo.late, reduce_C_function=np.mean, gridsize=55, cmap="Reds", mincnt=30, vmin=0, vmax=0.25)
axs[1].set_title("Late rate by area (cells with 30+ orders)"); fig.colorbar(hb2, ax=axs[1], fraction=0.04, label="late share (0.10 = 10%)")
for a in axs: a.set_xlabel("longitude"); a.set_ylabel("latitude")
plt.tight_layout(); plt.show()
say("Orders concentrate in the south-east, but the highest late rates appear along the north-east coast and around Rio de Janeiro / Espírito Santo. A reliable late-rate map needs enough orders per area, which is why small cells are hidden.")

## 4. Payments and approval
*Question: does the payment method, or a slow approval, relate to late delivery? (Your question about "approval delay as a reason".)*

In [ ]:
t_pay = rate_by(D, "pay_type", 20)
pay_share = D.pay_type.value_counts(normalize=True) * 100
res["share_boleto"] = float(pay_share.get("boleto", np.nan)); res["share_card"] = float(pay_share.get("credit_card", np.nan))
res["late_boleto"] = float(t_pay.late_pct.get("boleto", np.nan)); res["late_card"] = float(t_pay.late_pct.get("credit_card", np.nan))
fig, axs = plt.subplots(1, 3, figsize=(17, 4.4))
axs[0].bar(pay_share.index, pay_share.values, color=C_OK); axs[0].set_title("Share of orders by main payment type (%)"); axs[0].tick_params(axis="x", rotation=20)
bar_rate(axs[1], t_pay, "Late rate by payment type", base=BASE, rot=20)
pm = df[df.pay_type.notna()].groupby(["p_month", "pay_type"]).size().unstack(fill_value=0)
pm = pm[(pm.sum(axis=1) >= 500)]
pm = pm.div(pm.sum(axis=1), axis=0) * 100
axs[2].stackplot(pm.index, pm.T.values, labels=pm.columns, alpha=0.9)
axs[2].legend(loc="upper left", fontsize=7); axs[2].set_title("Payment mix over time (%)"); axs[2].tick_params(axis="x", rotation=70)
plt.tight_layout(); plt.show()
say(f"Credit card dominates ({res['share_card']:.0f}% of orders) and boleto is {res['share_boleto']:.0f}%. Late rates are close ({res['late_card']:.1f}% card vs {res['late_boleto']:.1f}% boleto), so payment type is not a direct cause; its real effect is on approval time (next cells).")

In [ ]:
D["appr_bin"] = pd.cut(D.appr_h, [-1, 1, 6, 24, 48, 1e5], labels=["<1h", "1-6h", "6-24h", "1-2d", ">2d"])
fig, axs = plt.subplots(1, 2, figsize=(14, 4.4))
for p_, c_ in [("credit_card", C_OK), ("boleto", C_ACC), ("debit_card", C_GREEN)]:
    x_ = np.sort(D.loc[D.pay_type == p_, "appr_h"].dropna().clip(upper=96).values)
    axs[0].plot(x_, np.arange(1, len(x_) + 1) / len(x_) * 100, label=p_, color=c_, lw=2)
axs[0].set_title("How fast are orders approved? (cumulative %)"); axs[0].set_xlabel("hours from purchase to approval"); axs[0].set_ylabel("% approved by then"); axs[0].legend()
t_appr = rate_by(D, "appr_bin", 500)
bar_rate(axs[1], t_appr, "Late rate by approval lag", base=BASE)
plt.tight_layout(); plt.show()
ap = D.groupby("pay_type").appr_h.agg(median="median", gt24=lambda s: (s > 24).mean() * 100)
res.update(med_appr_boleto=float(ap.loc["boleto", "median"]), med_appr_card_min=float(ap.loc["credit_card", "median"] * 60),
           gt24_boleto=float(ap.loc["boleto", "gt24"]), gt24_card=float(ap.loc["credit_card", "gt24"]),
           late_fast=float(D[D.appr_h < 1].late.mean() * 100), late_slow=float(D[D.appr_h > 24].late.mean() * 100),
           auc_appr=auc_uni(D.appr_h, D.late))
display(ap.rename(columns={"median": "median hours", "gt24": "% taking >24h"}))
say(f"Boleto approvals take a median {res['med_appr_boleto']:.0f} hours vs about {res['med_appr_card_min']:.0f} minutes for cards. Orders approved in under an hour are late {res['late_fast']:.1f}% of the time vs {res['late_slow']:.1f}% beyond 24h: a real but small effect (AUC {res['auc_appr']:.2f}).")

In [ ]:
slow = D[D.appr_h > 24]
res["slow_share_boleto"] = float((slow.pay_type == "boleto").mean() * 100); res["boleto_share_all"] = float((D.pay_type == "boleto").mean() * 100)
D["appr_speed"] = pd.cut(D.appr_h, [-1, 1, 24, 1e5], labels=["<1h", "1-24h", ">24h"])
two = D[D.pay_type.isin(["credit_card", "boleto"])]
px = two.groupby(["pay_type", "appr_speed"], observed=True).late.agg(n="size", late_pct="mean"); px["late_pct"] *= 100
D["post_appr"] = (D.order_delivered_carrier_date - D.order_approved_at).dt.total_seconds() / 86400
pa = D.groupby("appr_speed", observed=True).post_appr.median()
res["post_appr_fast"] = float(pa.get("<1h", np.nan)); res["post_appr_slow"] = float(pa.get(">24h", np.nan))
res["late_boleto_slow"] = float(px.late_pct.get(("boleto", ">24h"), np.nan)); res["late_card_slow"] = float(px.late_pct.get(("credit_card", ">24h"), np.nan))
res["late_boleto_fast"] = float(px.late_pct.get(("boleto", "<1h"), np.nan)); res["late_card_fast"] = float(px.late_pct.get(("credit_card", "<1h"), np.nan))
fig, axs = plt.subplots(1, 2, figsize=(14, 4.4))
pv = px.late_pct.unstack("pay_type"); pv.plot.bar(ax=axs[0], color=[C_ACC, C_OK], width=0.75)
axs[0].set_title("Late rate: payment type x approval speed"); axs[0].set_ylabel("Late rate (%)"); axs[0].tick_params(axis="x", rotation=0)
axs[1].bar(pa.index.astype(str), pa.values, color=C_GREY); axs[1].set_title("Median days from approval to carrier handoff"); axs[1].set_ylabel("days")
plt.tight_layout(); plt.show()
display(px.rename(columns={"n": "orders", "late_pct": "late %"}))
say(f"{res['slow_share_boleto']:.0f}% of slow approvals are boleto although boleto is only {res['boleto_share_all']:.0f}% of orders, so payment type explains WHY approval is slow. Slow approval goes with more late orders for both methods (boleto {res['late_boleto_fast']:.1f}% to {res['late_boleto_slow']:.1f}%, card {res['late_card_fast']:.1f}% to {res['late_card_slow']:.1f}%), so it is the delay itself that relates to lateness. Sellers do not take longer after a slow approval ({res['post_appr_slow']:.1f} vs {res['post_appr_fast']:.1f} days).")

## 5. Why three scoring stages?
*Question: what does each checkpoint of an order add? This section is the evidence for re-scoring at **purchase**, **approval** and **carrier handoff**.*
No model is trained here. We use descriptive late rates and one-column-at-a-time AUC.

In [ ]:
Dc = D[D.order_delivered_carrier_date.notna() & D.order_approved_at.notna()].copy()
Dc["t_appr"] = Dc.appr_h / 24; Dc["t_hand"] = Dc.handling_d; Dc["t_deliv"] = Dc.actual_d
med = Dc.groupby("late")[["t_appr", "t_hand", "t_deliv", "promise_d"]].median()
fig, ax = plt.subplots(figsize=(13, 4.8))
for yv, key, colr, name in [(1, 1.0, C_LATE, "late orders"), (0, 0.0, C_OK, "on-time orders")]:
    r = med.loc[key]
    ax.plot([0, r.t_deliv], [yv, yv], "-", color=colr, lw=3, zorder=1)
    ax.scatter([0, r.t_appr, r.t_hand, r.t_deliv], [yv] * 4, s=[120, 50, 120, 120], color=colr, zorder=3)
    ax.scatter([r.promise_d], [yv], marker="|", s=700, color="black", zorder=4)
    ax.text(-1.0, yv, name, ha="right", va="center", fontsize=10, fontweight="bold", color=colr)
    ax.annotate("SCORE 1: order placed\nSCORE 2: approved (hours later)", (0, yv), xytext=(0.3, yv + 0.30), fontsize=8.5, ha="left", va="bottom")
    ax.annotate(f"SCORE 3: handed to carrier\n(median day {r.t_hand:.1f})", (r.t_hand, yv), xytext=(r.t_hand + 0.3, yv - 0.30), fontsize=8.5, ha="left", va="top",
                arrowprops=dict(arrowstyle="-", color=C_GREY))
    ax.annotate(f"delivered: label known\n(median day {r.t_deliv:.1f})", (r.t_deliv, yv), xytext=(r.t_deliv + 0.4, yv - 0.30), fontsize=8.5, ha="left", va="top",
                arrowprops=dict(arrowstyle="-", color=C_GREY))
ax.annotate("| = promised date", (med.loc[0.0].promise_d, 0), xytext=(med.loc[0.0].promise_d, 0.28), ha="center", fontsize=8)
ax.set_xlim(-10, 42); ax.set_ylim(-0.8, 1.7); ax.set_yticks([]); ax.set_xlabel("median days since purchase")
ax.set_title("An order's journey (medians): what becomes known when")
plt.tight_layout(); plt.show()
display(med.rename(columns={"t_appr": "approved (days)", "t_hand": "handoff (days)", "t_deliv": "delivered (days)", "promise_d": "promised (days)"}).rename(index={0.0: "on time", 1.0: "late"}))
say("Approval and handoff happen within the first days, while delivery takes weeks. Re-scoring at those two checkpoints lets the system update its view long before the outcome is known.")

In [ ]:
st3 = Dc.groupby("late")[["handling_d", "transit_d", "promise_d"]].median().rename(index={0.0: "on time", 1.0: "late"})
ax = st3.T.plot.bar(figsize=(9, 4.2), color=[C_OK, C_LATE]); ax.set_title("Median days per stage: late vs on-time orders"); ax.set_ylabel("days"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
res.update(handling_ok=float(st3.loc["on time", "handling_d"]), handling_late=float(st3.loc["late", "handling_d"]),
           transit_ok=float(st3.loc["on time", "transit_d"]), transit_late=float(st3.loc["late", "transit_d"]),
           promise_ok=float(st3.loc["on time", "promise_d"]), promise_late=float(st3.loc["late", "promise_d"]))
display(st3)
say(f"Late orders spend {res['transit_late']:.0f} days in carrier transit versus {res['transit_ok']:.0f} for on-time ones, while the promised window is the same ({res['promise_late']:.1f} vs {res['promise_ok']:.1f} days). Lateness is mostly a transit problem; seller handling adds a smaller share ({res['handling_late']:.1f} vs {res['handling_ok']:.1f} days).")

In [ ]:
Dc["overrun_bin"] = pd.cut(Dc.seller_overrun_d, [-1e3, -5, -1, 0, 2, 5, 1e3], labels=["5+ d early", "1-5 d early", "0-1 d early", "0-2 d late", "2-5 d late", "5+ d late"])
t_ov = rate_by(Dc, "overrun_bin", 300)
mm_ = Dc.groupby(Dc.seller_overrun_d > 0).late.mean() * 100
res.update(late_met_limit=float(mm_.loc[False]), late_missed_limit=float(mm_.loc[True]), late_over5=float(t_ov.late_pct.iloc[-1]),
           auc_overrun=auc_uni(Dc.seller_overrun_d, Dc.late), auc_handling=auc_uni(Dc.handling_d, Dc.late))
fig, ax = plt.subplots(figsize=(10, 4.2))
bar_rate(ax, t_ov, "Late rate by how the seller did against its own shipping deadline", base=BASE)
plt.tight_layout(); plt.show()
say(f"When the carrier receives the parcel we learn whether the seller met its deadline: orders are late {res['late_met_limit']:.1f}% if it did and {res['late_missed_limit']:.1f}% if not, and {res['late_over5']:.0f}% if the seller was more than 5 days over. This is the strongest signal in the data and exists only at checkpoint 3.")

In [ ]:
p90 = Dc[Dc.late == 0].transit_d.quantile(0.9)
Lt = Dc[Dc.late == 1].copy()
Lt["reason"] = np.where(Lt.seller_overrun_d > 0, "seller missed shipping limit",
                np.where(Lt.transit_d > p90, f"slow transit (> {p90:.0f} d = p90 of on-time)", "other"))
sh = Lt.reason.value_counts(normalize=True) * 100
res.update(attr_transit=float(sh.iloc[[i for i, k in enumerate(sh.index) if k.startswith("slow")][0]]),
           attr_seller=float(sh.get("seller missed shipping limit", 0)),
           attr_other=float(sh.get("other", 0)),
           ontime_missed=float((Dc[Dc.late == 0].seller_overrun_d > 0).mean() * 100),
           late_missed=float((Lt.seller_overrun_d > 0).mean() * 100))
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.barh(sh.index[::-1], sh.values[::-1], color=[C_GREY, C_ACC, C_LATE])
for i, v in enumerate(sh.values[::-1]): ax.text(v + 0.5, i, f"{v:.0f}%", va="center")
ax.set_title("Rule-based attribution of late orders (a stage attribution, not a proven cause)"); ax.set_xlabel("% of late orders")
plt.tight_layout(); plt.show()
say(f"{res['attr_transit']:.0f}% of late orders show unusually slow transit, {res['attr_seller']:.0f}% had the seller miss its limit, {res['attr_other']:.0f}% neither. Seller misses are {res['late_missed']:.0f}% of late orders but only {res['ontime_missed']:.0f}% of on-time ones. The data cannot say WHY transit was slow (no carrier ID, scans or weather), so these are attributions, not causes.")

In [ ]:
# How much does each checkpoint add? one-column-at-a-time AUC, grouped by when the column becomes known
stage_feats = {
    "1 purchase": ["promise_d", "dist_km", "same_state", "freight", "price", "weight_g", "vol_cm3", "n_items", "installments", "p_hour", "p_dow"],
    "2 approval": ["appr_h"],
    "3 handoff": ["handling_d", "seller_overrun_d", "post_appr_d"],
}
rows = []
for stg, cols_ in stage_feats.items():
    for c_ in cols_:
        a_ = auc_uni(D[c_], D.late)
        rows.append((stg, c_, a_, max(a_, 1 - a_)))
auc_tbl = pd.DataFrame(rows, columns=["stage", "feature", "auc_raw", "strength"]).sort_values(["stage", "strength"], ascending=[True, False])
res["auc"] = dict(zip(auc_tbl.feature, auc_tbl.auc_raw))
res["best_purchase_auc"] = float(auc_tbl[auc_tbl.stage == "1 purchase"].strength.max())
res["best_handoff_auc"] = float(auc_tbl[auc_tbl.stage == "3 handoff"].strength.max())
colmap = {"1 purchase": C_OK, "2 approval": C_ACC, "3 handoff": C_LATE}
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(auc_tbl.feature[::-1], auc_tbl.strength[::-1] - 0.5, left=0.5, color=[colmap[s] for s in auc_tbl.stage[::-1]])
ax.axvline(0.5, c="black", lw=1); ax.set_xlim(0.45, 0.7)
ax.set_title("Single-column strength vs 'late' (AUC; 0.5 = no signal)"); ax.set_xlabel("AUC (direction ignored)")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=v, label=k) for k, v in colmap.items()], loc="lower right")
plt.tight_layout(); plt.show()
display(auc_tbl.rename(columns={"auc_raw": "AUC (raw)", "strength": "strength"}).reset_index(drop=True))
say(f"The best purchase-time column reaches AUC {res['best_purchase_auc']:.2f}, approval adds a small signal (AUC {res['auc_appr']:.2f}), and handoff columns reach {res['best_handoff_auc']:.2f}. Each checkpoint knows more than the one before, so the same order's risk estimate sharpens as it progresses. The approval step adds the least; we report that honestly.")

In [ ]:
# Same purchase-time risk, very different outcomes once the handoff is known
H = Dc[Dc.seller_overrun_d.notna()].copy()
H["route_risk"] = H.groupby("customer_state").late.transform("mean")                      # purchase-time knowledge only
H["risk_tier"] = pd.qcut(H.route_risk.rank(method="first"), 3, labels=["low-risk route", "mid-risk route", "high-risk route"])
H["handoff"] = pd.cut(H.seller_overrun_d, [-1e3, -3, 0, 2, 1e3], labels=["3+ d early", "0-3 d early", "0-2 d late", "2+ d late"])
hm = H.groupby(["risk_tier", "handoff"], observed=True).late.agg(n="size", p="mean")
pv_ = (hm.p.unstack() * 100); nv_ = hm.n.unstack()
lab_ = pv_.round(1).astype(str) + "%\n(n=" + nv_.astype(int).map("{:,}".format) + ")"
fig, ax = plt.subplots(figsize=(10, 4.2))
sns.heatmap(pv_, annot=lab_, fmt="", cmap="Reds", cbar_kws={"label": "late rate (%)"}, ax=ax, linewidths=1)
ax.set_title("Late rate by route risk (known at purchase) x seller timing (known at handoff)"); ax.set_xlabel("seller vs its shipping deadline"); ax.set_ylabel("")
plt.tight_layout(); plt.show()
res["heat_min"] = float(pv_.min().min()); res["heat_max"] = float(pv_.max().max())
res["heat_mid_lo"] = float(pv_.loc["mid-risk route"].min()); res["heat_mid_hi"] = float(pv_.loc["mid-risk route"].max())
say(f"Orders on the same mid-risk route range from {res['heat_mid_lo']:.1f}% to {res['heat_mid_hi']:.1f}% late once we know how the seller performed. A purchase-time score cannot see that, which is exactly why we re-score at handoff.")

## 6. Time and drift
*Question: is lateness stable over time? (It decides the train/test split and whether month can be a feature.)*

In [ ]:
mo = D.groupby("p_month").late.agg(n="size", late_pct="mean"); mo["late_pct"] *= 100
mo = mo[mo.n >= 500]
fig, ax = plt.subplots(figsize=(13, 4.6)); ax2 = ax.twinx()
ax2.bar(mo.index, mo.n, color=C_GREY, alpha=0.35); ax2.set_ylabel("orders", color=C_GREY); ax2.grid(False)
ax.plot(mo.index, mo.late_pct, marker="o", color=C_LATE, lw=2); ax.set_ylabel("Late rate (%)", color=C_LATE)
ax.set_title("Late rate by purchase month (bars = order volume)"); ax.tick_params(axis="x", rotation=60); ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)
for m_ in ["2017-11", "2018-02", "2018-03"]:
    if m_ in mo.index: ax.annotate(f"{mo.late_pct[m_]:.1f}%", (m_, mo.late_pct[m_]), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9, color=C_LATE)
plt.tight_layout(); plt.show()
for m_ in ["2017-02", "2017-03", "2017-11", "2018-02", "2018-03"]:
    res["month_" + m_.replace("-", "_")] = float(mo.late_pct.get(m_, np.nan))
res["month_2017_min"] = float(mo[mo.index < "2018-01"].late_pct.min()); res["month_2017_min_name"] = mo[mo.index < "2018-01"].late_pct.idxmin()
say(f"Lateness is not stable: from {res['month_2017_min']:.1f}% ({res['month_2017_min_name']}) to {res['month_2017_11']:.1f}% in Nov 2017 (Black Friday) and {res['month_2018_03']:.1f}% in March 2018. The system must cope with sudden surges and the dashboard should show them.")

In [ ]:
D["regime"] = np.where(D.p_month.isin(["2017-11", "2018-02", "2018-03"]), "surge months (Nov17, Feb18, Mar18)", "other months")
sv = D.groupby("regime").agg(orders=("late", "size"), late_pct=("late", lambda s: s.mean() * 100),
                             handling_d=("handling_d", "median"), transit_d=("transit_d", "median"), promise_d=("promise_d", "median"),
                             seller_missed_pct=("seller_overrun_d", lambda s: (s.dropna() > 0).mean() * 100))
display(sv.T)
fig, ax = plt.subplots(figsize=(9, 4))
sv[["transit_d", "handling_d"]].T.plot.bar(ax=ax, color=[C_ACC, C_GREY]); ax.set_title("Median days per stage: surge vs other months"); ax.set_ylabel("days"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
sg = "surge months (Nov17, Feb18, Mar18)"
res.update(transit_surge=float(sv.loc[sg, "transit_d"]), transit_norm=float(sv.loc["other months", "transit_d"]),
           handling_surge=float(sv.loc[sg, "handling_d"]), handling_norm=float(sv.loc["other months", "handling_d"]),
           missed_surge=float(sv.loc[sg, "seller_missed_pct"]), missed_norm=float(sv.loc["other months", "seller_missed_pct"]))
say(f"In surge months carrier transit stretches from {res['transit_norm']:.1f} to {res['transit_surge']:.1f} days, while seller handling moves only from {res['handling_norm']:.1f} to {res['handling_surge']:.1f}. The surge is mostly carrier capacity, which no purchase-time feature describes.")

In [ ]:
yo = D[D.p_month.isin(["2017-02", "2018-02", "2017-03", "2018-03"])].copy()
yo["year"] = yo.p_month.str[:4]; yo["mon"] = yo.p_month.str[5:]
yo_t = yo.groupby(["mon", "year"]).late.mean().unstack() * 100
display(yo_t.rename(index={"02": "February", "03": "March"}))
splits = {"train (2017-01..2017-12)": ("2017-01-01", "2018-01-01"), "validation (2018-02..2018-04)": ("2018-02-01", "2018-05-01"),
          "test (2018-05..2018-08)": ("2018-05-01", "2018-09-01")}
rows = []
for k, (a_, b_) in splits.items():
    s_ = D[(D.order_purchase_timestamp >= a_) & (D.order_purchase_timestamp < b_)]
    rows.append((k, len(s_), s_.late.mean() * 100)); res[f"split_{k.split()[0]}_n"] = len(s_); res[f"split_{k.split()[0]}_late"] = s_.late.mean() * 100
sp = pd.DataFrame(rows, columns=["set", "orders", "late %"]).set_index("set"); display(sp)
fig, ax = plt.subplots(figsize=(13, 4.2))
colors_ = [C_OK if m_ < "2018-01" else (C_GREY if m_ == "2018-01" else (C_ACC if m_ < "2018-05" else C_GREEN)) for m_ in mo.index]
ax.bar(mo.index, mo.late_pct, color=colors_); ax.tick_params(axis="x", rotation=60)
ax.set_title("Chronological split: blue = train, grey = gap month, orange = validation, green = test"); ax.set_ylabel("Late rate (%)")
plt.tight_layout(); plt.show()
res["feb17"], res["mar17"] = float(yo_t.loc["02", "2017"]), float(yo_t.loc["03", "2017"])
res["feb18"], res["mar18"] = float(yo_t.loc["02", "2018"]), float(yo_t.loc["03", "2018"])
say(f"Same calendar month, different regime: Feb/Mar 2017 were {res['feb17']:.1f}%/{res['mar17']:.1f}% late and Feb/Mar 2018 {res['feb18']:.1f}%/{res['mar18']:.1f}%. That is a regime shift, not seasonality, so month and year are NOT model features and we split chronologically (a random split would hide the drift). Because base rates differ between sets, we judge alerts by top-k risk rather than a fixed probability.")

## 7. Sellers
*Question: are some sellers consistently late, and is that stable enough to use?*

In [ ]:
sl = D.groupby("seller_id").late.agg(n="size", rate="mean"); sl["rate"] *= 100
sl50 = sl[sl.n >= 50]
q10, q50, q90 = np.percentile(sl50.rate, [10, 50, 90])
res.update(sl_p10=float(q10), sl_p50=float(q50), sl_p90=float(q90), sl_n50=len(sl50), sl_over15=int((sl50.rate > 15).sum()),
           sellers_total=len(sl), orders_per_seller_median=float(sl.n.median()), sellers_lt10=float((sl.n < 10).mean() * 100))
fig, axs = plt.subplots(1, 2, figsize=(14, 4.2))
axs[0].hist(sl50.rate, bins=30, color=C_OK); axs[0].axvline(BASE, c=C_GREY, ls="--")
axs[0].set_title(f"Late rate per seller ({len(sl50)} sellers with 50+ orders)"); axs[0].set_xlabel("Late rate (%)")
axs[1].hist(sl.n, bins=np.logspace(0, np.log10(sl.n.max()), 40), color=C_ACC); axs[1].set_xscale("log")
axs[1].set_title("Orders per seller (log scale)"); axs[1].set_xlabel("orders")
plt.tight_layout(); plt.show()
say(f"Sellers differ: among those with 50+ orders the 10th/50th/90th percentile late rates are {q10:.1f}% / {q50:.1f}% / {q90:.1f}%, and {res['sl_over15']} sellers exceed 15%. But the median seller has only {res['orders_per_seller_median']:.0f} orders and {res['sellers_lt10']:.0f}% have fewer than 10, so a raw seller_id column would just memorise noise.")

In [ ]:
D["half"] = np.where(D.order_purchase_timestamp < "2018-01-01", "H1", "H2")


def persistence(key, minn):
    p = D.groupby([key, "half"]).late.agg(["size", "mean"]).unstack("half")
    ok = (p[("size", "H1")] >= minn) & (p[("size", "H2")] >= minn)
    q = p[ok]
    return spearmanr(q[("mean", "H1")], q[("mean", "H2")])[0], int(ok.sum()), q


pers = {}
for key, mn in [("seller_id", 30), ("customer_state", 200), ("category_en", 200), ("seller_state", 200)]:
    r_, n_, q_ = persistence(key, mn); pers[key] = (r_, n_, q_)
res.update(persist_seller=float(pers["seller_id"][0]), persist_state=float(pers["customer_state"][0]), persist_cat=float(pers["category_en"][0]))
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, key, ttl in [(axs[0], "seller_id", "sellers (30+ orders in each half)"), (axs[1], "customer_state", "customer states")]:
    q_ = pers[key][2]
    ax.scatter(q_[("mean", "H1")] * 100, q_[("mean", "H2")] * 100, color=C_OK, alpha=0.7)
    ax.set_xlabel("late rate in 2017 (%)"); ax.set_ylabel("late rate in 2018 (%)"); ax.set_title(f"{ttl}: rank corr {pers[key][0]:.2f}")
plt.tight_layout(); plt.show()
display(pd.DataFrame({k: {"rank correlation": v[0], "groups compared": v[1]} for k, v in pers.items()}).T)
say(f"A group's late rate in 2017 predicts its 2018 rate: states {res['persist_state']:.2f}, sellers {res['persist_seller']:.2f}, categories {res['persist_cat']:.2f}. Sellers are partly persistent (so history helps) but noisy (so it is only a modest signal).")

In [ ]:
# Leakage-safe seller history: for each order use ONLY this seller's orders delivered BEFORE the order was placed
S = D[(D.order_status == "delivered") & (D.n_sellers == 1) & D.order_delivered_carrier_date.notna()].copy()
S = S.sort_values("order_purchase_timestamp").reset_index(drop=True)
S["overrun"] = (S.seller_overrun_d > 0).astype(float)
n_prior = np.zeros(len(S)); late_prior = np.zeros(len(S)); over_prior = np.zeros(len(S)); hand_sum = np.zeros(len(S))
to_ns = lambda s_: s_.values.astype("datetime64[ns]").astype("int64")
for sid, g in S.groupby("seller_id"):
    idx = g.index.values
    dt = to_ns(g.order_delivered_customer_date); o_ = np.argsort(dt); dts = dt[o_]
    cl = np.cumsum(g.late.values[o_]); co = np.cumsum(g.overrun.values[o_]); ch = np.cumsum(g.handling_d.values[o_])
    k = np.searchsorted(dts, to_ns(g.order_purchase_timestamp), side="left")     # prior orders already delivered
    n_prior[idx] = k
    for arr, c_ in ((late_prior, cl), (over_prior, co), (hand_sum, ch)):
        arr[idx] = np.where(k > 0, c_[np.maximum(k - 1, 0)], 0)
S["n_prior"] = n_prior; S["late_prior_cnt"] = late_prior
a_, base_ = 10, 0.07
S["seller_late_prior"] = (late_prior + a_ * base_) / (n_prior + a_)
S["seller_overrun_prior"] = (over_prior + a_ * 0.08) / (n_prior + a_)
S["seller_handling_prior"] = np.where(n_prior > 0, hand_sum / np.maximum(n_prior, 1), np.nan)
S["seller_late_raw"] = np.where(n_prior > 0, late_prior / np.maximum(n_prior, 1), np.nan)
test_m = (S.order_purchase_timestamp >= "2018-05-01") & (S.order_purchase_timestamp < "2018-09-01")
T = S[test_m]
sh_feats = ["seller_late_prior", "seller_overrun_prior", "seller_handling_prior", "n_prior"]
sh_auc = pd.DataFrame({"all orders": [auc_uni(S[c], S.late) for c in sh_feats], "test period (May-Aug 2018)": [auc_uni(T[c], T.late) for c in sh_feats]}, index=sh_feats)
display(sh_auc)
T20 = T[T.n_prior >= 20]
qn = pd.qcut(T20.seller_late_prior, 5, duplicates="drop")
qt = T20.groupby(qn, observed=True).late.agg(n="size", late_pct="mean"); qt["late_pct"] *= 100
fig, ax = plt.subplots(figsize=(9, 4))
bar_rate(ax, qt, "Test-period late rate by the seller's PAST late rate (fifths; sellers with 20+ prior orders)", base=T20.late.mean() * 100, fs=8)
ax.tick_params(axis="x", labelsize=7); plt.tight_layout(); plt.show()
res.update(sh_auc_late=float(sh_auc.loc["seller_late_prior", "test period (May-Aug 2018)"]),
           sh_auc_over=float(sh_auc.loc["seller_overrun_prior", "test period (May-Aug 2018)"]),
           sh_cov5=float((T.n_prior >= 5).mean() * 100), sh_none=float((T.n_prior == 0).mean() * 100),
           sh_q_first=float(qt.late_pct.iloc[0]), sh_q_last=float(qt.late_pct.iloc[-1]))
say(f"Using only past information, seller history ranks test-period orders only slightly better than chance (AUC {res['sh_auc_late']:.2f} and {res['sh_auc_over']:.2f}). Still, the late rate rises from {res['sh_q_first']:.1f}% to {res['sh_q_last']:.1f}% across fifths, and {res['sh_cov5']:.0f}% of test orders have a seller with 5+ prior orders ({res['sh_none']:.1f}% have none). We include it as 4 smoothed features, not as raw seller_id.")

## 8. Label quality: multi-seller orders
*Question: is "late" trustworthy for orders shipped by several sellers?*

In [ ]:
D["sellers_bin"] = D.n_sellers.clip(upper=3).astype(int).astype(str).replace({"3": "3+"})
t_ms = rate_by(D, "sellers_bin", 1)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
bar_rate(axs[0], t_ms, "Late rate by number of sellers in the order", base=BASE)
bar_rate(axs[1], rate_by(D, "items_bin", 1), "Late rate by number of items", base=BASE)
plt.tight_layout(); plt.show()
res.update(late_1seller=float(t_ms.late_pct.get("1", np.nan)), late_2seller=float(t_ms.late_pct.get("2", np.nan)), n_multiseller=int((D.n_sellers > 1).sum()))
say(f"Multi-seller orders look almost never late ({res['late_2seller']:.1f}% vs {res['late_1seller']:.1f}% for single-seller orders). That is suspicious, not good news: the recorded delivery time seems to mark the first parcel. Section 10 shows customers of these orders complain about missing items far more often. We therefore exclude the {res['n_multiseller']:,} multi-seller orders from training and evaluation.")

## 9. Correlations and overlap
*Question: which signals are strongest, and which carry the same information?*

In [ ]:
all_feats = {"promise_d": "purchase", "dist_km": "purchase", "same_state": "purchase", "freight": "purchase", "freight_ratio": "purchase",
             "price": "purchase", "weight_g": "purchase", "vol_cm3": "purchase", "max_len": "purchase", "n_items": "purchase",
             "n_sellers": "purchase", "n_products": "purchase", "photos": "purchase", "name_len": "purchase", "desc_len": "purchase",
             "limit_slack_d": "purchase", "installments": "purchase", "pay_value": "purchase", "pay_n": "purchase",
             "p_hour": "purchase", "p_dow": "purchase", "appr_h": "approval", "handling_d": "handoff",
             "seller_overrun_d": "handoff", "post_appr_d": "handoff"}
rows = []
for c_, s_ in all_feats.items():
    ok = D[c_].notna() & D.days_late.notna()
    a_ = auc_uni(D[c_], D.late)
    rows.append((c_, s_, a_, max(a_, 1 - a_), spearmanr(D[c_][ok], D.days_late[ok])[0], D[c_].isna().mean() * 100))
ft = pd.DataFrame(rows, columns=["feature", "known at", "AUC (raw)", "strength", "Spearman vs days_late", "% null"]).sort_values("strength", ascending=False).reset_index(drop=True)
display(ft)
fig, ax = plt.subplots(figsize=(10, 8))
cm_ = {"purchase": C_OK, "approval": C_ACC, "handoff": C_LATE}
ax.barh(ft.feature[::-1], ft.strength[::-1] - 0.5, left=0.5, color=[cm_[s] for s in ft["known at"][::-1]])
ax.axvline(0.5, c="black", lw=1); ax.set_xlim(0.47, 0.7); ax.set_title("All candidate columns ranked by single-column strength (AUC vs late)")
ax.legend(handles=[Patch(color=v, label=k) for k, v in cm_.items()], loc="lower right")
plt.tight_layout(); plt.show()
res["auc_table"] = dict(zip(ft.feature, ft["AUC (raw)"]))
res["spearman_promise_days_late"] = float(ft.loc[ft.feature == "promise_d", "Spearman vs days_late"].iloc[0])
weak = ft[(ft.strength < 0.515) & (ft["known at"] == "purchase")].feature.tolist()
dropped = [w for w in weak if DICT.get(w, ("",))[0] in ("Excluded", "Filter")]
kept = [w for w in weak if w not in dropped]
say(f"Near 0.5 on their own: {', '.join(weak)}. Of these, {', '.join(dropped) or 'none'} are dropped; {', '.join(kept) or 'none'} stay only because they are free at checkout and a tree model can use them in combination. Note that promise_d correlates {res['spearman_promise_days_late']:.2f} with days_late only because days_late is delivered minus promised: that is arithmetic, not insight, which is why we rank against the late flag instead.")

In [ ]:
pc = ["dist_km", "same_state", "freight", "weight_g", "vol_cm3", "price", "promise_d", "n_items"]
cm = D[pc].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title("Spearman correlation between purchase-time features")
plt.tight_layout(); plt.show()
res.update(corr_dist_same=float(cm.loc["dist_km", "same_state"]), corr_dist_promise=float(cm.loc["dist_km", "promise_d"]),
           corr_dist_freight=float(cm.loc["dist_km", "freight"]), corr_weight_vol=float(cm.loc["weight_g", "vol_cm3"]),
           corr_freight_weight=float(cm.loc["freight", "weight_g"]))
say(f"Many signals overlap: distance vs same-state {res['corr_dist_same']:.2f}, distance vs promised window {res['corr_dist_promise']:.2f}, distance vs freight {res['corr_dist_freight']:.2f}, weight vs volume {res['corr_weight_vol']:.2f}. Olist's promise already encodes the route, and freight already carries weight and distance, so these strengths do not add up.")

## 10. Customer reviews (post-delivery analysis only)
*Question: do reviews explain late deliveries, and how do they relate to real lateness?*
Reviews are written **after** delivery, so they are **never** a model input. Here they are used to validate our delay reasons and to show what customers experience.
Comments are in Brazilian Portuguese, so we work with accent-stripped keyword rules (and, adapted from the reference notebook, a regex text-cleaning step and n-grams).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

REV = find_file(["olist_order_reviews_dataset.csv", "olist_order_reviews_dataset*.zip", "olist_order_reviews_dataset*.csv"])
rv = pd.read_csv(REV)


def norm(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s.lower())
    return "".join(c for c in s if not unicodedata.combining(c))


rv["txt"] = (rv.review_comment_title.fillna("") + " . " + rv.review_comment_message.fillna("")).map(norm)
rv["has_text"] = rv.txt.str.strip(". ") != ""
sc = rv.review_score.value_counts(normalize=True).sort_index() * 100
tx = rv.groupby("review_score").has_text.mean() * 100
res.update(rev_rows=len(rv), rev_orders=rv.order_id.nunique(), text_cov=float(rv.has_text.mean() * 100), five_star=float(sc.loc[5]),
           multi_review_orders=int((rv.groupby("order_id").size() > 1).sum()))
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].bar(sc.index.astype(str), sc.values, color=[C_LATE, C_LATE, C_ACC, C_OK, C_OK]); axs[0].set_title("Review score distribution (%)"); axs[0].set_xlabel("stars")
axs[1].bar(tx.index.astype(str), tx.values, color=C_GREY); axs[1].set_title("Share of reviews that include a comment, by score (%)"); axs[1].set_xlabel("stars")
plt.tight_layout(); plt.show()
print(f"reviews: {len(rv):,} on {rv.order_id.nunique():,} orders ({res['multi_review_orders']} orders have more than one review)")
say(f"{res['text_cov']:.0f}% of reviews contain a comment (unhappy customers write more), and {res['five_star']:.1f}% are 5 stars. Text exists for only a minority of orders, so reviews can explain some late orders but never all of them.")

In [ ]:
# Text preparation, adapted from the reference notebook (regex cleaning), kept light on purpose


def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"[\r\n]+", " ", s)                       # line breaks
    s = re.sub(r"https?://\S+|www\.\S+", " ", s)         # links
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"r\$\s?\d+[\d\.,]*", " ", s)             # money
    s = re.sub(r"\d+", " ", s)                           # numbers
    s = re.sub(r"[^a-z\s]", " ", s)                      # punctuation / symbols
    return re.sub(r"\s+", " ", s).strip()


msg = rv.review_comment_message.where(rv.has_text)
rv["clean"] = msg.fillna("").map(clean_text)
ex = rv[(rv.review_score <= 2) & rv.has_text].sample(5, random_state=7)[["review_comment_message"]].copy()
ex["after cleaning"] = ex.review_comment_message.map(clean_text)
display(ex.rename(columns={"review_comment_message": "original (Portuguese)"}))
say("Cleaning strips line breaks, links, numbers, accents and punctuation. Accent stripping matters: 'não' and 'nao' must match the same pattern.")

In [ ]:
LATE_PAT = (r"atras|demor|fora do prazo|passou do prazo|prazo (?:ja )?(?:venceu|expirou|esgot)|nao chegou|"
            r"nao recebi|ainda nao (?:chegou|recebi|foi entregue|entregaram)|nunca chegou|"
            r"ate (?:agora|o momento|hoje).{0,25}nao|aguardando|esperando")
NEG_OK = r"(?:nao|sem|nenhum)\s+(?:houve\s+)?(?:atras|demor)"
NEVER_PAT = (r"nao (?:chegou|recebi)|nunca chegou|ainda nao (?:chegou|recebi|foi entregue|entregaram)|"
             r"ate (?:agora|o momento|hoje).{0,25}nao|nao foi entregue")
SLOW_PAT = r"atras|demor|fora do prazo|passou do prazo|chegou (?:depois|tarde)|veio (?:depois|tarde)|entregue (?:depois|tarde)"
PARTIAL_PAT = (r"so recebi|apenas (?:um|uma|1)|somente (?:um|uma)|faltou|faltando|veio so|chegou so|so chegou|so veio|"
               r"recebi (?:apenas|somente|so|parte)|entreg(?:ue|aram) (?:apenas|so|somente)")
CAUSES = {
    "carrier / Correios / tracking": r"correio|transportadora|rastre|codigo de rastreio|extravi|perdid|entregador|motoboy|greve|sedex",
    "seller slow to ship / dispatch": (r"vendedor.{0,60}(?:postou|enviou|despach|demor)|(?:postou|postagem|despach).{0,40}(?:atras|demor|dias|tarde)|"
                                       r"nota fiscal|nf-?e|nao (?:foi )?(?:enviad|postad|despachad)|ainda nao (?:enviou|postou|enviaram|postaram)|"
                                       r"aguardando (?:o )?envio|sem (?:envio|postagem)"),
    "wrong / incorrect address": r"endereco (?:errad|incorret|inexist|nao encontr)|nao (?:localiz|encontr)ou? o endereco|cep (?:errad|incorret)|ausente|ninguem (?:em casa|para receber)|tentativa de entrega",
    "no communication / no support": r"nenhum (?:contato|retorno|resposta|posicionamento|aviso|informacao)|sem (?:retorno|resposta|contato|informacao|satisfacao)|ninguem (?:responde|me respond|retorn|ligou)|nao (?:responde|retorn|tive (?:retorno|resposta))|descaso",
    "refund / cancel requested": r"estorn|extorn|reembols|cancel|dinheiro de volta|devolv(?:a|am|er) (?:o )?(?:meu )?(?:dinheiro|valor)",
    "out of stock": r"sem estoque|falta de estoque|estoque|indisponivel",
    "marked delivered but not received": r"(?:consta|informa|diz|aparece|mostra|site informa|sistema).{0,40}(?:entregue|recebi)|constando (?:como )?entregue",
}
rules_tbl = pd.DataFrame({"rule": ["delay / non-arrival complaint", "never arrived / still waiting", "arrived slow or past deadline", "partial / missing item"] + list(CAUSES),
                          "example Portuguese keywords (accent-free)": ["atras*, demor*, fora do prazo, nao chegou, aguardando", "nao chegou, nao recebi, ainda nao recebi, nunca chegou",
                                                                        "atras*, demor*, passou do prazo, chegou depois", "so recebi, apenas um, faltou, veio so",
                                                                        "correio, transportadora, rastreio, extravi*", "vendedor ... postou/enviou, nota fiscal, nao foi enviado",
                                                                        "endereco errado, cep incorreto, ausente", "sem retorno, ninguem responde, descaso",
                                                                        "estorno, reembolso, cancel*", "sem estoque, indisponivel", "consta entregue, site informa entregue"]})
display(rules_tbl)
rv["complaint"] = rv.txt.str.contains(LATE_PAT, regex=True) & ~rv.txt.str.contains(NEG_OK, regex=True)
rv["never"] = rv.txt.str.contains(NEVER_PAT, regex=True)
rv["slow"] = rv.txt.str.contains(SLOW_PAT, regex=True)
rv["partial"] = rv.txt.str.contains(PARTIAL_PAT, regex=True)
rv["carrier"] = rv.txt.str.contains(CAUSES["carrier / Correios / tracking"], regex=True)
rv["seller"] = rv.txt.str.contains(CAUSES["seller slow to ship / dispatch"], regex=True)
lc = rv[rv.has_text & rv.complaint]
res.update(rev_complaints=len(lc), rev_complaint_pct_text=float(len(lc) / rv.has_text.sum() * 100), rev_complaint_pct_all=float(len(lc) / len(rv) * 100),
           rev_complaint_score=float(lc.review_score.mean()), rev_complaint_1star=float((lc.review_score == 1).mean() * 100),
           rev_never=int(lc.txt.str.contains(NEVER_PAT, regex=True).sum()), rev_slow=int(lc.txt.str.contains(SLOW_PAT, regex=True).sum()),
           rev_partial=int(lc.txt.str.contains(PARTIAL_PAT, regex=True).sum()))
print(f"reviews with a delay / non-arrival complaint: {len(lc):,}  ({res['rev_complaint_pct_text']:.1f}% of commented reviews, {res['rev_complaint_pct_all']:.1f}% of all reviews)")
print(f"  average score {res['rev_complaint_score']:.2f}; {res['rev_complaint_1star']:.0f}% give 1 star")
print(f"  never arrived / still waiting: {res['rev_never']:,} | arrived slow: {res['rev_slow']:,} | partial: {res['rev_partial']:,}   (groups overlap)")
say(f"About {len(lc):,} reviews complain about lateness or non-arrival, and they score {res['rev_complaint_score']:.1f} stars on average. These are keyword rules, so counts are estimates. Negations such as 'sem atraso' (no delay) are excluded.")

In [ ]:
# Join reviews to orders (one row per order) and compare with real lateness
G = rv.assign(rdate=pd.to_datetime(rv.review_creation_date)).groupby("order_id").agg(
    min_score=("review_score", "min"), complaint=("complaint", "max"), never=("never", "max"), partial=("partial", "max"),
    carrier=("carrier", "max"), seller=("seller", "max"), has_text=("has_text", "max"), rdate=("rdate", "min")).reset_index()
R = D.merge(G, on="order_id", how="inner")
R["late_bucket"] = pd.cut(R.days_late, [-100, -0.0001, 3, 7, 14, 1000], labels=["on time / early", "0-3 d late", "3-7 d late", "7-14 d late", ">14 d late"])
bk = R.groupby("late_bucket", observed=True).agg(orders=("order_id", "size"), avg_score=("min_score", "mean"),
                                                 pct_1_2_stars=("min_score", lambda s: (s <= 2).mean() * 100),
                                                 pct_complaint=("complaint", lambda s: s.mean() * 100))
lv = R.groupby("late").agg(orders=("order_id", "size"), avg_score=("min_score", "mean"), pct_1_2_stars=("min_score", lambda s: (s <= 2).mean() * 100),
                           pct_complaint=("complaint", lambda s: s.mean() * 100)).rename(index={0.0: "on time", 1.0: "late"})
display(lv); display(bk)
res.update(score_ok=float(lv.loc["on time", "avg_score"]), score_late=float(lv.loc["late", "avg_score"]),
           p12_ok=float(lv.loc["on time", "pct_1_2_stars"]), p12_late=float(lv.loc["late", "pct_1_2_stars"]),
           cmp_ok=float(lv.loc["on time", "pct_complaint"]), cmp_late=float(lv.loc["late", "pct_complaint"]),
           bk_03=float(bk.loc["0-3 d late", "avg_score"]), bk_37=float(bk.loc["3-7 d late", "avg_score"]),
           p12_03=float(bk.loc["0-3 d late", "pct_1_2_stars"]), p12_37=float(bk.loc["3-7 d late", "pct_1_2_stars"]),
           bk_7p=float(R[R.days_late > 7].min_score.mean()), p12_7p=float((R[R.days_late > 7].min_score <= 2).mean() * 100))
fig, axs = plt.subplots(1, 2, figsize=(14, 4.2))
axs[0].bar(bk.index.astype(str), bk.avg_score, color=[C_OK, C_ACC, C_LATE, C_LATE, C_LATE]); axs[0].set_title("Average review score by how late the order was"); axs[0].set_ylim(1, 5); axs[0].tick_params(axis="x", rotation=20)
axs[1].bar(bk.index.astype(str), bk.pct_1_2_stars, color=[C_OK, C_ACC, C_LATE, C_LATE, C_LATE]); axs[1].set_title("% of 1-2 star reviews by lateness"); axs[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()
say(f"Lateness is what customers punish: {res['score_ok']:.2f} stars on time vs {res['score_late']:.2f} when late ({res['p12_ok']:.0f}% vs {res['p12_late']:.0f}% give 1-2 stars). Up to 3 days late is tolerated ({res['bk_03']:.1f} stars); beyond 3 days satisfaction collapses. That supports a 'severely late' (3+ days) alert level.")

In [ ]:
c_ = R.complaint.astype(bool)
res["cmp_precision"] = float(R[c_].late.mean() * 100); res["cmp_recall"] = float(c_[R.late == 1].mean() * 100); res["cmp_base"] = float(R.late.mean() * 100)
R["ym"] = R.order_purchase_timestamp.dt.to_period("M").astype(str)
mmn = R.groupby("ym").agg(n=("order_id", "size"), late=("late", lambda s: s.mean() * 100), cmp=("complaint", lambda s: s.mean() * 100))
mmn = mmn[mmn.n > 500]
res["cmp_month_corr"] = float(mmn.late.corr(mmn.cmp))
fig, axs = plt.subplots(1, 2, figsize=(14, 4.2))
axs[0].plot(mmn.index, mmn.late, marker="o", color=C_LATE, label="real late rate (%)"); axs[0].plot(mmn.index, mmn.cmp, marker="s", color=C_OK, label="reviews complaining about delay (%)")
axs[0].legend(); axs[0].tick_params(axis="x", rotation=60); axs[0].set_title(f"Monthly late rate vs complaint rate (correlation {res['cmp_month_corr']:.2f})")
axs[1].bar(["share of complaining\norders truly late", "base late rate", "share of all late orders\nthat complain"], [res["cmp_precision"], res["cmp_base"], res["cmp_recall"]], color=[C_OK, C_GREY, C_ACC])
axs[1].set_title("How good is a delay complaint as a lateness signal?"); axs[1].set_ylabel("%")
plt.tight_layout(); plt.show()
say(f"A delay complaint is a strong but incomplete signal: {res['cmp_precision']:.0f}% of complaining orders were truly late (base rate {res['cmp_base']:.1f}%), yet only {res['cmp_recall']:.0f}% of late orders produce such a complaint. Complaint rate follows the monthly late rate closely ({res['cmp_month_corr']:.2f}), so reviews are a good lagging indicator on the dashboard but cannot replace the label.")

In [ ]:
cause_n = {k: int(lc.txt.str.contains(p, regex=True).sum()) for k, p in CAUSES.items()}
named = pd.concat([lc.txt.str.contains(p, regex=True) for p in CAUSES.values()], axis=1).any(axis=1)
res.update(causes=cause_n, cause_named=int(named.sum()), cause_named_pct=float(named.mean() * 100))
ct = pd.Series(cause_n).sort_values()
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.barh(ct.index, ct.values, color=C_LATE)
for i, v in enumerate(ct.values): ax.text(v + 5, i, f"{v:,} ({v / len(lc) * 100:.0f}%)", va="center", fontsize=9)
ax.set_title(f"Causes named inside {len(lc):,} delay complaints (a review can match several)"); ax.set_xlabel("reviews")
plt.tight_layout(); plt.show()
say(f"Only {res['cause_named_pct']:.0f}% of delay complaints name any cause; the rest just say the order did not arrive. When a cause is named, the carrier / postal service is mentioned far more than the seller, which matches the data-side attribution (slow transit is the larger share of late orders). Wrong address is almost never mentioned. These are what customers say, not verified causes.")

In [ ]:
# What words do unhappy vs happy customers use? (n-grams, adapted from the reference notebook)
PT_STOP = set("""de a o que e do da em um para com uma os no se na por mais as dos como mas foi ao ele das tem seu sua ou ser quando muito ja eu tambem so pelo pela ate isso ela entre era depois sem mesmo aos ter seus quem nas me esse eles estao voce tinha foram essa num nem suas meu minha numa pelos elas havia seja qual sera nos tenho lhe deles essas esses pelas este fosse dele tu te voces vos lhes meus minhas teu tua nosso nossa dela delas esta estes estas aquele aquela isto aquilo estou estamos pra produto comprei compra recebi lannister targaryen stark""".split())
low = rv[rv.has_text & (rv.review_score <= 2)].clean
high = rv[rv.has_text & (rv.review_score >= 4)].clean


def top_bigrams(texts, k=15):
    cv = CountVectorizer(ngram_range=(2, 2), stop_words=list(PT_STOP), min_df=5)
    X = cv.fit_transform(texts)
    cnt = np.asarray(X.sum(axis=0)).ravel()
    order = cnt.argsort()[::-1][:k]
    names = np.array(cv.get_feature_names_out())
    return pd.Series(cnt[order], index=names[order])[::-1]


bl, bh = top_bigrams(low), top_bigrams(high)
fig, axs = plt.subplots(1, 2, figsize=(14, 5.5))
axs[0].barh(bl.index, bl.values, color=C_LATE); axs[0].set_title("Top word pairs in 1-2 star reviews")
axs[1].barh(bh.index, bh.values, color=C_OK); axs[1].set_title("Top word pairs in 4-5 star reviews")
plt.tight_layout(); plt.show()
delay_pairs = ["nao chegou", "nao recebi", "ainda nao", "prazo entrega", "nao foi entregue", "entrega atrasou", "atraso entrega"]
share_pairs = {p: float(low.str.contains(p).mean() * 100) for p in delay_pairs}
display(pd.Series(share_pairs, name="% of 1-2 star comments containing it").to_frame())
res["top_low_bigram"] = bl.index[-1]
say(f"The leading word pairs among unhappy customers are about non-arrival ('{bl.index[-1]}', '{bl.index[-2]}', '{bl.index[-3]}'), while happy customers praise early delivery and quality ('{bh.index[-1]}', '{bh.index[-2]}'). This independent view supports our keyword rules and shows why 'nao' (not) must be kept, not treated as a stop-word.")

In [ ]:
# Multi-seller orders: customers say parts of the order are missing, while 'late' stays low
X = df.merge(G, on="order_id", how="left")
R["sellers_bin"] = R.n_sellers.clip(upper=3).astype(int).astype(str).replace({"3": "3+"})
pt = R.groupby("sellers_bin").agg(orders=("order_id", "size"), late_pct=("late", lambda s: s.mean() * 100),
                                  partial_complaint_pct=("partial", lambda s: s.fillna(False).astype(bool).mean() * 100))
display(pt)
res.update(partial_1=float(pt.loc["1", "partial_complaint_pct"]), partial_2=float(pt.loc["2", "partial_complaint_pct"]),
           partial_3=float(pt.loc["3+", "partial_complaint_pct"]) if "3+" in pt.index else np.nan)
nv = X[X["never"].fillna(False).astype(bool)]
nvd = nv[nv.delivered]
res.update(never_orders=len(nv), never_with_date=float(nv.delivered.mean() * 100),
           never_before_delivery=float((nvd.rdate < nvd.order_delivered_customer_date.dt.normalize()).mean() * 100))
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].bar(pt.index, pt.partial_complaint_pct, color=C_ACC); axs[0].set_title("% of orders whose review says only part arrived"); axs[0].set_xlabel("sellers in the order")
sv_ = nv.order_status.value_counts()
axs[1].bar(sv_.index, sv_.values, color=C_GREY); axs[1].set_title(f"'Never arrived' reviews by order status (n={len(nv):,})"); axs[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
print(f"'never arrived' reviews: {len(nv):,} orders | {res['never_with_date']:.0f}% have a delivery date | {res['never_before_delivery']:.0f}% of those were written BEFORE the delivery date")
say(f"Customers of multi-seller orders report missing items {res['partial_2']:.0f}% of the time vs {res['partial_1']:.1f}% for single-seller orders, although these orders show almost no lateness. That confirms the 'late' label is unreliable for them. Also, {res['never_before_delivery']:.0f}% of 'never arrived' reviews were written before the parcel actually arrived: they mean 'not yet'.")

## 11. Rules for the stream (R1 to R9)
*Question: which simple, explainable checks are worth running next to the model, and how much do they actually catch?*
A rule is useful when (a) it fires reasonably often, (b) orders it flags are clearly later than the rest (**lift**), and (c) it catches a meaningful share of late orders (**coverage**).
Rules are evaluated on delivered orders where the information needed by the rule exists.

In [ ]:
def eval_rule(name, definition, known_at, base, mask):
    mask = mask.reindex(base.index).fillna(False).astype(bool)
    f = base[mask]; nf = base[~mask]
    lf, ln = f.late.mean() * 100, nf.late.mean() * 100
    return dict(rule=name, definition=definition, known_at=known_at, evaluated_on=len(base), fires_pct=mask.mean() * 100,
                late_when_fired=lf, late_otherwise=ln, lift=lf / ln if ln else np.nan, late_orders_caught_pct=(mask & (base.late == 1)).sum() / (base.late == 1).sum() * 100)


D8 = D.copy()
sm = D.groupby(["customer_state", "p_month"]).late.agg(prev_n="size", prev_rate="mean").reset_index()
sm["p_month"] = (pd.PeriodIndex(sm.p_month, freq="M") + 1).astype(str)
D8 = D8.merge(sm, on=["customer_state", "p_month"], how="left"); D8.index = D.index
base1 = D[D.appr_h.notna()]
base2 = Dc
base3 = Dc[Dc.transit_d.notna()]
base5 = D
base6 = D[D.dist_km.notna()]
base7 = S[S.n_prior >= 20]
base8 = D8[D8.prev_n >= 100]
rows = [
    eval_rule("R1 slow approval", "approval takes more than 24 h", "approval", base1, base1.appr_h > 24),
    eval_rule("R2 seller missed deadline", "carrier handoff after shipping_limit_date", "handoff", base2, base2.seller_overrun_d > 0),
    eval_rule("R3 slow transit", f"carrier transit above {p90:.0f} days (p90 of on-time orders)", "in transit", base3, base3.transit_d > p90),
    eval_rule("R5 tight promise", "promised window of 10 days or less", "purchase", base5, base5.promise_d <= 10),
    eval_rule("R6 long route", "seller-to-customer distance over 1,000 km", "purchase", base6, base6.dist_km > 1000),
    eval_rule("R7 unreliable seller", "seller's PAST late rate above 15% (20+ prior orders)", "purchase", base7, base7.seller_late_raw > 0.15),
    eval_rule("R8 region spike", "customer state's late rate LAST month at least 15% (100+ orders)", "purchase", base8, base8.prev_rate >= 0.15),
]
rt = pd.DataFrame(rows).set_index("rule")
stuck_row = pd.DataFrame([dict(definition="promised date passed and no delivery event", known_at="after promise", evaluated_on=res["stuck_n"], fires_pct=100.0,
                               late_when_fired=np.nan, late_otherwise=np.nan, lift=np.nan, late_orders_caught_pct=np.nan)], index=["R4 already late / stuck (no label)"])
r9 = pd.DataFrame([dict(definition="model score in the top-k% (after training)", known_at="each stage", evaluated_on=np.nan, fires_pct=np.nan,
                        late_when_fired=np.nan, late_otherwise=np.nan, lift=np.nan, late_orders_caught_pct=np.nan)], index=["R9 high model risk (evaluated after training)"])
rt_all = pd.concat([rt.iloc[:3], stuck_row, rt.iloc[3:], r9])
display(rt_all.style.format({"fires_pct": "{:.1f}", "late_when_fired": "{:.1f}", "late_otherwise": "{:.1f}", "lift": "{:.2f}", "late_orders_caught_pct": "{:.1f}", "evaluated_on": "{:,.0f}"}, na_rep="-"))
for k_, nm in [("R1", "R1 slow approval"), ("R2", "R2 seller missed deadline"), ("R3", "R3 slow transit"), ("R5", "R5 tight promise"),
               ("R6", "R6 long route"), ("R7", "R7 unreliable seller"), ("R8", "R8 region spike")]:
    res[f"rule_{k_}_fired_late"] = float(rt.loc[nm, "late_when_fired"]); res[f"rule_{k_}_lift"] = float(rt.loc[nm, "lift"]); res[f"rule_{k_}_caught"] = float(rt.loc[nm, "late_orders_caught_pct"])
wk = [res["rule_R1_lift"], res["rule_R5_lift"], res["rule_R6_lift"], res["rule_R7_lift"]]
say(f"R2 (seller missed its shipping deadline) is the sharpest rule usable at handoff: lift {res['rule_R2_lift']:.1f}x, flagged orders are late {res['rule_R2_fired_late']:.0f}% of the time, and it catches {res['rule_R2_caught']:.0f}% of late orders. R8 (state surge) comes next (lift {res['rule_R8_lift']:.1f}x). R3 looks stronger (lift {res['rule_R3_lift']:.0f}x, {res['rule_R3_caught']:.0f}% caught) but is partly circular: a parcel in transit longer than 90% of on-time parcels is almost by definition late, so R3 is a live monitor for parcels in transit, not a prediction. R1, R5, R6 and R7 are weak alone (lift {min(wk):.1f}x to {max(wk):.1f}x), including the past-seller-record rule R7; we keep them as explainable alert reasons, not stand-alone predictors. R4 covers the {res['stuck_n']:,} unlabeled undelivered orders; R9 is the model alert (top-k% risk), evaluated after training.")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 4.6))
names = list(rt.index)
x_ = np.arange(len(names))
axs[0].bar(x_ - 0.2, rt.late_when_fired, width=0.4, color=C_LATE, label="late % when rule fires")
axs[0].bar(x_ + 0.2, rt.late_otherwise, width=0.4, color=C_OK, label="late % otherwise")
axs[0].axhline(BASE, ls="--", c=C_GREY); axs[0].set_xticks(x_); axs[0].set_xticklabels([n_.split(" ")[0] for n_ in names]); axs[0].legend(); axs[0].set_title("Do flagged orders end up later?")
axs[1].bar(x_, rt.late_orders_caught_pct, color=C_ACC)
for i, (c_, f_) in enumerate(zip(rt.late_orders_caught_pct, rt.fires_pct)): axs[1].text(i, c_ + 1, f"{c_:.0f}%\n(fires {f_:.0f}%)", ha="center", fontsize=8)
axs[1].set_xticks(x_); axs[1].set_xticklabels([n_.split(" ")[0] for n_ in names]); axs[1].set_title("Share of late orders each rule catches"); axs[1].set_ylim(0, rt.late_orders_caught_pct.max() * 1.3)
plt.tight_layout(); plt.show()
say("Each rule trades precision against coverage. In the stream they run alongside the model, and each alert shows its reason, which a probability alone cannot do.")

In [ ]:
# R8: state x month late rate (where the surge hits)
tops = D.customer_state.value_counts().head(12).index
hmx = D[D.customer_state.isin(tops) & (D.p_month >= "2017-01") & (D.p_month <= "2018-08")].groupby(["customer_state", "p_month"]).late.agg(n="size", p="mean").reset_index()
pv2 = hmx.pivot(index="customer_state", columns="p_month", values="p") * 100
pn2 = hmx.pivot(index="customer_state", columns="p_month", values="n")
pv2 = pv2.where(pn2 >= 30)
fig, ax = plt.subplots(figsize=(15, 5.5))
sns.heatmap(pv2, cmap="Reds", annot=True, fmt=".0f", annot_kws={"size": 7}, cbar_kws={"label": "late rate (%)"}, ax=ax, linewidths=0.3)
ax.set_title("Late rate (%) by customer state and purchase month (cells with <30 orders hidden)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()
m3 = pv2["2018-03"].dropna().sort_values(ascending=False)
say(f"The early-2018 surge hits most states at once, but not equally: in March 2018 {', '.join(f'{k_} {v_:.0f}%' for k_, v_ in m3.head(4).items())} are hit hardest while SP stays at {m3.get('SP', float('nan')):.0f}%. A rolling per-state late-rate window (rule R8) lets the dashboard show this live.")

## 12. Conclusions
*Question: do the numbers quoted in the project discussion reproduce from the data, and what design follows?*

In [ ]:
checks = [  # (label, expected value quoted earlier, key in res or callable, tolerance)
    ("Orders in master", 99441, "rows", 0), ("Columns in master", 51, "cols", 0), ("Delivered orders", 96476, "delivered", 0),
    ("Late orders", 6535, "late_n", 0), ("On-time orders", 89941, "ontime_n", 0), ("Not delivered", 2965, "undelivered_n", 0),
    ("Late rate % (date rule)", 6.77, "late_rate", 0.02), ("Late rate % (naive timestamp rule)", 8.1, "ts_late_rate", 0.1),
    ("Orders whose label differs between the two rules", 1292, "label_differ", 0), ("Median days late (date level)", 7.5, "median_days_late", 0.6),
    ("Undelivered orders already past promise", 2961, "stuck_n", 0), ("Orders with >1 item row %", 9.9, "multi_item", 0.3),
    ("Orders with >1 payment row %", 3.0, "multi_pay", 0.5), ("Orders with no items", 775, "no_items", 0),
    ("Handoff before approval", 1350, "carrier_before_appr", 0), ("Handoff before purchase", 165, "carrier_before_purchase", 0),
    ("Delivered before handoff", 23, "deliv_before_carrier", 0), ("Category missing %", 1.4, "null_category", 0.2), ("Distance missing %", 0.5, "null_dist", 0.1),
    ("Late % distance <100 km", 4.5, "dist_first", 0.3), ("Late % distance >1,500 km", 11.6, "dist_last", 0.3),
    ("Late % same state", 4.5, "same_rate", 0.3), ("Late % cross state", 8.0, "cross_rate", 0.3),
    ("Late % state PR", 4.0, "rate_PR", 0.3), ("Late % state SP", 4.5, "rate_SP", 0.3), ("Late % state CE", 13.8, "rate_CE", 0.3), ("Late % state MA", 17.4, "rate_MA", 0.3),
    ("600-1500 km: SP late %", 3.3, "ctrl_SP", 0.5), ("600-1500 km: MG late %", 3.3, "ctrl_MG", 0.5),
    ("600-1500 km: RJ late %", 11.5, "ctrl_RJ", 1.0), ("600-1500 km: BA late %", 11.5, "ctrl_BA", 1.0), ("600-1500 km: ES late %", 11.5, "ctrl_ES", 1.0),
    ("Late % freight <=10", 4.3, "fr_first", 0.3), ("Late % freight >50", 8.6, "fr_last", 0.3),
    ("Late % weight <=300 g", 6.2, "w_first", 0.3), ("Late % weight >10 kg", 8.9, "w_last", 0.3),
    ("Lowest category late %", 4.3, "cat_min", 0.4), ("Highest category late %", 8.2, "cat_max", 0.4),
    ("Late % promise <=10 d", 10.5, "promise_short", 0.4), ("Late % promise >40 d", 2.3, "promise_long", 0.4),
    ("Boleto median approval (h)", 29, "med_appr_boleto", 2), ("Card median approval (min)", 16, "med_appr_card_min", 2),
    ("Boleto approvals >24 h %", 57, "gt24_boleto", 2), ("Card approvals >24 h %", 7, "gt24_card", 1.5),
    ("Late % approved <1 h", 6.3, "late_fast", 0.2), ("Late % approved >24 h", 8.2, "late_slow", 0.4),
    ("Slow approvals that are boleto %", 65, "slow_share_boleto", 2), ("Boleto share of orders %", 20, "boleto_share_all", 1),
    ("Post-approval handoff days (slow approval)", 1.5, "post_appr_slow", 0.2), ("Post-approval handoff days (fast approval)", 1.9, "post_appr_fast", 0.2),
    ("Transit days late orders", 26.1, "transit_late", 0.3), ("Transit days on-time", 6.9, "transit_ok", 0.3),
    ("Handling days late orders", 3.5, "handling_late", 0.3), ("Handling days on-time", 2.1, "handling_ok", 0.3),
    ("Promise days late orders", 22.5, "promise_late", 0.3), ("Promise days on-time", 23.3, "promise_ok", 0.3),
    ("Late % seller met limit", 5.4, "late_met_limit", 0.3), ("Late % seller missed limit", 20.8, "late_missed_limit", 0.4), ("Late % seller >5 d over", 49, "late_over5", 2),
    ("Attribution: slow transit %", 66, "attr_transit", 1.5), ("Attribution: seller missed %", 28, "attr_seller", 1.5), ("Attribution: other %", 6, "attr_other", 1.5),
    ("On-time orders where seller missed limit %", 7.7, "ontime_missed", 0.5),
    ("AUC distance", 0.58, lambda: res["auc"]["dist_km"], 0.01), ("AUC freight", 0.56, lambda: res["auc"]["freight"], 0.01),
    ("AUC approval lag", 0.536, lambda: res["auc"]["appr_h"], 0.01), ("AUC handling time", 0.651, lambda: res["auc"]["handling_d"], 0.01),
    ("AUC seller overrun", 0.657, lambda: res["auc"]["seller_overrun_d"], 0.01), ("AUC promise window", 0.474, lambda: res["auc"]["promise_d"], 0.01),
    ("AUC weight", 0.519, lambda: res["auc"]["weight_g"], 0.01),
    ("Late % Nov 2017", 12.3, "month_2017_11", 0.5), ("Late % Feb 2017", 3.0, "feb17", 0.3), ("Late % Mar 2017", 4.6, "mar17", 0.3),
    ("Late % Feb 2018", 14.1, "feb18", 0.3), ("Late % Mar 2018", 19.0, "mar18", 0.4),
    ("Surge months: transit days", 9.8, "transit_surge", 0.3), ("Other months: transit days", 6.9, "transit_norm", 0.3),
    ("Surge months: handling days", 2.8, "handling_surge", 0.3), ("Other months: handling days", 2.1, "handling_norm", 0.3),
    ("Train orders", 43426, "split_train_n", 0), ("Train late %", 5.6, "split_train_late", 0.1),
    ("Validation orders", 20357, "split_validation_n", 0), ("Validation late %", 12.6, "split_validation_late", 0.1),
    ("Test orders", 25352, "split_test_n", 0), ("Test late %", 4.4, "split_test_late", 0.1),
    ("Seller late % p10", 1.6, "sl_p10", 0.3), ("Seller late % p50", 5.7, "sl_p50", 0.3), ("Seller late % p90", 12.0, "sl_p90", 0.5),
    ("Sellers with 50+ orders", 419, "sl_n50", 0), ("...of which over 15% late", 18, "sl_over15", 0),
    ("Median orders per seller", 7, "orders_per_seller_median", 0.5), ("Sellers with <10 orders %", 59, "sellers_lt10", 2),
    ("Persistence: sellers", 0.35, "persist_seller", 0.05), ("Persistence: states", 0.62, "persist_state", 0.05), ("Persistence: categories", 0.32, "persist_cat", 0.05),
    ("Seller-history AUC (past late rate)", 0.52, "sh_auc_late", 0.015), ("Seller-history AUC (past overrun rate)", 0.53, "sh_auc_over", 0.015),
    ("Test orders with seller 5+ prior orders %", 87, "sh_cov5", 1), ("Test orders with no seller history %", 4.4, "sh_none", 0.5),
    ("Late % lowest fifth by seller history", 4.1, "sh_q_first", 0.5), ("Late % highest fifth by seller history", 5.6, "sh_q_last", 0.5),
    ("Late % single-seller orders", 6.7, "late_1seller", 0.3), ("Late % two-seller orders", 1.0, "late_2seller", 0.3),
    ("Corr distance vs same-state", -0.76, "corr_dist_same", 0.03), ("Corr distance vs promise", 0.62, "corr_dist_promise", 0.03),
    ("Corr distance vs freight", 0.58, "corr_dist_freight", 0.03), ("Corr weight vs volume", 0.78, "corr_weight_vol", 0.03), ("Corr freight vs weight", 0.49, "corr_freight_weight", 0.03),
    ("Reviews with text %", 43, "text_cov", 1), ("5-star reviews %", 57.8, "five_star", 0.3),
    ("Delay complaints (count)", 5100, "rev_complaints", 300), ("Delay complaints avg score", 1.8, "rev_complaint_score", 0.1),
    ("Score: on-time orders", 4.29, "score_ok", 0.03), ("Score: late orders", 2.27, "score_late", 0.03),
    ("1-2 stars % on-time", 9, "p12_ok", 1), ("1-2 stars % late", 62, "p12_late", 1.5),
    ("Complaint % on-time", 2.5, "cmp_ok", 0.3), ("Complaint % late", 32, "cmp_late", 1.5),
    ("Score 0-3 d late", 3.8, "bk_03", 0.1), ("Score 3-7 d late", 2.3, "bk_37", 0.1), ("Score 7+ d late", 1.7, "bk_7p", 0.1),
    ("1-2 stars % 0-3 d late", 19, "p12_03", 1.5), ("1-2 stars % 3-7 d late", 61, "p12_37", 1.5), ("1-2 stars % 7+ d late", 78, "p12_7p", 2),
    ("Complaint precision %", 48, "cmp_precision", 1.5), ("Complaint recall %", 32, "cmp_recall", 1.5), ("Monthly late vs complaint corr", 0.96, "cmp_month_corr", 0.02),
    ("Complaints naming a cause %", 22, "cause_named_pct", 1.5),
    ("Cause: carrier (count)", 490, lambda: res["causes"]["carrier / Correios / tracking"], 40), ("Cause: seller slow (count)", 180, lambda: res["causes"]["seller slow to ship / dispatch"], 20),
    ("Cause: no communication (count)", 140, lambda: res["causes"]["no communication / no support"], 20), ("Cause: marked delivered (count)", 100, lambda: res["causes"]["marked delivered but not received"], 20),
    ("Cause: out of stock (count)", 40, lambda: res["causes"]["out of stock"], 15), ("Cause: wrong address (count)", 4, lambda: res["causes"]["wrong / incorrect address"], 3),
    ("Partial-item complaints % (1 seller)", 1.6, "partial_1", 0.3), ("Partial-item complaints % (2 sellers)", 24, "partial_2", 1.5),
    ("'Never arrived' reviews (orders)", 3538, "never_orders", 30), ("...with a delivery date %", 80, "never_with_date", 2), ("...written before delivery %", 54, "never_before_delivery", 2),
    ("Rule R2: late % when fired", 20.8, "rule_R2_fired_late", 0.4),
]
out = []
for label, exp, src, tol in checks:
    try:
        got = src() if callable(src) else res[src]
        ok = abs(got - exp) <= tol
        out.append((label, exp, round(float(got), 3), tol, "PASS" if ok else "CHECK"))
    except Exception as e:
        out.append((label, exp, np.nan, tol, "N/A"))
chk = pd.DataFrame(out, columns=["claim", "quoted earlier", "recomputed here", "tolerance", "status"])
n_pass = int((chk.status == "PASS").sum())
print(f"{n_pass} of {len(chk)} quoted numbers reproduce within tolerance.")
print("Rows marked CHECK differ from the figure quoted in the project discussion; the value recomputed here is the one to use.")
display(chk[chk.status != "PASS"] if (chk.status != "PASS").any() else pd.DataFrame({"result": ["all quoted numbers reproduce"]}))
with pd.option_context("display.max_rows", 300):
    display(chk.style.apply(lambda r: ["background-color:#d6f5d6" if r.status == "PASS" else "background-color:#fde2e2"] * len(r), axis=1))

In [ ]:
dec = pd.DataFrame([
    ("Prediction target", "late = delivered date > promised date (date-level); binary; trained on delivered, single-seller orders"),
    ("Why not a delay-in-days target", f"promised dates are day-level and delays have a long tail; a days target mostly echoes the promise window (Spearman {res['spearman_promise_days_late']:.2f})"),
    ("Scoring point 1: order placed", f"15 purchase-time features + 4 seller-history features (best single column AUC {res['best_purchase_auc']:.2f})"),
    ("Scoring point 2: order approved", f"adds approval lag (AUC {res['auc_appr']:.2f}); small gain, reported honestly"),
    ("Scoring point 3: handed to carrier", f"adds handling time, shipping-limit overrun, approval-to-handoff (AUC up to {res['best_handoff_auc']:.2f}); strongest checkpoint"),
    ("Dropped features", "month/year (regime shift), city/zip (too many values), raw seller_id, photo/name/description length, price ratios, duplicate counts"),
    ("Seller information", f"as past-only history (4 smoothed features) and rule R7; raw seller_id kept only as a key and for the dashboard"),
    ("Split", f"chronological: train 2017 ({res['split_train_n']:,}), validation Feb-Apr 2018 ({res['split_validation_n']:,}), test May-Aug 2018 ({res['split_test_n']:,}); one month gap"),
    ("Metrics", "PR-AUC, ROC-AUC, recall in the top 10% riskiest orders, per month; compare against a state + distance baseline; alert on top-k, not a fixed probability"),
    ("Stream rules", "R1 slow approval, R2 seller missed deadline, R3 slow transit, R4 already late/stuck, R5 tight promise, R6 long route, R7 unreliable seller, R8 region spike, R9 model top-k"),
    ("Reviews", "post-delivery analytics only: validate delay reasons, show complaint rate beside late rate; never a model input"),
    ("Exclusions", f"{res['n_multiseller']:,} multi-seller orders removed (label unreliable); {res['stuck_n']:,} undelivered orders handled by rule R4, not the model"),
    ("What replay cannot show", "live carrier tracking, weather, strikes, real load; delay 'reasons' are stage attributions, not proven causes"),
], columns=["decision", "evidence-backed choice"]).set_index("decision")
with pd.option_context("display.max_colwidth", 200):
    display(dec)
say("The data supports one clear objective (will the order be late?), three scoring checkpoints that each add information, a handful of explainable rules, and honest limits. These are the decisions carried into the Kafka pipeline.")

## 13. Model 1 training (purchase-time model)
*Question: how well can we predict, at the moment an order is placed, that it will be late?*

**Run this section in Google Colab or locally.** It is self-contained: it needs only `orders_clean.csv` (and optionally `feature_registry.json`). Upload them to the Colab file panel (left sidebar), then run the cells below in order. Nothing in this section has been run for you, so every number it prints comes from your own run.

**Data plan used here**

| Set | Built from | Rows | Late |
|---|---|---|---|
| **Training** | all 42,947 training-period rows (Jan-Dec 2017) **plus** the 2,555 late orders of Feb-Apr 2018, so **all 5,000 late orders** | 45,502 | 5,000 |
| **Validation** | only the **on-time** orders of Feb-Apr 2018 | 17,533 | 0 |
| **Test** | all real orders of May-Aug 2018 | 24,928 | 1,114 |
| Not used | January 2018 (a buffer month) | 6,967 | 403 |

**What this plan means**
* A validation set with no late orders **cannot rank models** (no ROC/PR is possible). We use it for one thing it can do well: setting the alert threshold so that only about 10% of on-time orders get flagged (the false-alarm budget).
* Choosing the model type therefore uses **forward-chaining cross-validation inside 2017** (train on the past, test on the next months), where late orders exist.
* The late orders of Feb-Apr 2018 enter training without their on-time neighbours from the same months. If this inflates risk for the 2018 surge regions, the false-alarm number on validation will show it. For the standard protocol set `USE_VALID_LATE_IN_TRAIN = False` in the config cell and re-run.
* The test set is touched only once, at the end.

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, confusion_matrix
from sklearn.inspection import permutation_importance
import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12, "figure.figsize": (10, 4.5),
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
SEED = 42
np.random.seed(SEED)
C_LATE, C_OK, C_GREY, C_ACC = "#d1495b", "#2e86ab", "#8d99ae", "#edae49"

CFG = dict(
    CLEAN_FILE="orders_clean.csv",
    REGISTRY_FILE="feature_registry.json",
    USE_VALID_LATE_IN_TRAIN=True,    # True = your plan (all 5,000 late orders in training, validation = on-time only)
    INCLUDE_GAP_IN_TEST=False,       # True adds January 2018 (6,967 rows, 403 late) to the test set
    ONTIME_PER_LATE=None,            # None = use every training row (class weights handle the imbalance); 2 = keep 1 late : 2 on-time
    FPR_TARGET=0.10,                 # alert budget: share of on-time orders we accept to flag
    TOP_K=0.10,                      # 'late orders caught in the top 10% riskiest'
    CV_MODELS=["logistic_regression", "hist_gradient_boosting", "random_forest"],
    FORCE_MODEL=None,                # e.g. "hist_gradient_boosting" to skip the automatic choice
)


def find_data_file(name, dirs=(".", "/content", "/content/drive/MyDrive", "data", "/mnt/user-data/uploads", "/mnt/user-data/outputs")):
    for d in dirs:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found. Upload it in the Colab file panel (left sidebar) or mount Google Drive.")


print("scikit-learn", sklearn.__version__)

In [ ]:
# Load orders_clean.csv: real, labelled, single-seller delivered orders (one row per order)
clean = pd.read_csv(find_data_file(CFG["CLEAN_FILE"]), parse_dates=["purchase_ts"])
try:
    reg = json.load(open(find_data_file(CFG["REGISTRY_FILE"])))
except FileNotFoundError:
    reg = None

FALLBACK = ["promise_days", "purchase_hour", "purchase_weekday", "price", "freight", "n_items", "weight_g", "volume_cm3", "category",
            "customer_state", "seller_state", "same_state", "dist_km", "pay_type", "installments", "seller_prior_orders",
            "seller_late_rate_prior", "seller_overrun_rate_prior", "seller_handling_days_prior",
            "route_recent_days", "promise_buffer_route", "promise_buffer_state", "order_load_ratio"]
FEATS = reg["models"]["model_1"]["features"] if reg else FALLBACK
CAT = [c for c in (reg["categorical_features"] if reg else ["category", "customer_state", "seller_state", "pay_type"]) if c in FEATS]
NUM = [c for c in FEATS if c not in CAT]
missing = [c for c in FEATS + ["is_late", "split", "order_id"] if c not in clean.columns]
assert not missing, f"columns missing from {CFG['CLEAN_FILE']}: {missing}"
print(f"Model 1 uses {len(FEATS)} columns known when the order is placed")
print("  numeric    :", NUM)
print("  categorical:", CAT)
t = clean.groupby("split").is_late.agg(rows="size", late="sum")
t["on_time"] = t.rows - t.late; t["late_%"] = t.late / t.rows * 100
display(t.loc[["train", "gap", "valid", "test"]])

In [ ]:
# Assemble the three sets according to the data plan
tr_main = clean[clean.split == "train"]
if CFG["USE_VALID_LATE_IN_TRAIN"]:
    TR = pd.concat([tr_main, clean[(clean.split == "valid") & (clean.is_late == 1)]])
    VA = clean[(clean.split == "valid") & (clean.is_late == 0)]
else:
    TR = tr_main
    VA = clean[clean.split == "valid"]
TE = clean[clean.split.isin(["test", "gap"] if CFG["INCLUDE_GAP_IN_TEST"] else ["test"])]
TR, VA, TE = TR.sort_values("purchase_ts"), VA.sort_values("purchase_ts"), TE.sort_values("purchase_ts")

assert not (set(TR.order_id) & set(VA.order_id)) and not (set(TR.order_id) & set(TE.order_id)) and not (set(VA.order_id) & set(TE.order_id)), "an order appears in two sets"
summ = pd.DataFrame({n: dict(rows=len(d), late=int(d.is_late.sum()), on_time=int((d.is_late == 0).sum()), late_pct=d.is_late.mean() * 100,
                              first_purchase=str(d.purchase_ts.min().date()), last_purchase=str(d.purchase_ts.max().date()))
                     for n, d in [("training", TR), ("validation", VA), ("test", TE)]}).T
display(summ)
print(f"training rows {len(TR):,} = {int(TR.is_late.sum()):,} late + {int((TR.is_late == 0).sum()):,} on-time")
if CFG["USE_VALID_LATE_IN_TRAIN"]:
    print("NOTE: validation contains on-time orders only, so it cannot compute ROC/PR; it is used to set the alert threshold.")

In [ ]:
# Feature preparation. The SAME code is saved to model1_inference.py and reused by the Kafka scorer (no training/serving mismatch).
INFERENCE_PY = '''
import numpy as np, pandas as pd, joblib


def load_bundle(path="model1_bundle.joblib"):
    return joblib.load(path)


def prepare_features(df, b):
    """df must contain the Model 1 columns (same names as orders_stream.csv). Returns the model matrix."""
    X = df[b["features"]].copy()
    if "category" in X.columns:
        X["category"] = X["category"].where(X["category"].isin(b["top_categories"]), "other")
    for c in b["categorical"]:
        X[c] = X[c].astype(str).map(b["codes"][c]).fillna(-1).astype(int)     # unseen value -> -1 (treated as missing)
    return X


def score_orders(df, b):
    """Returns (risk score, alert flag). Score is a ranking score, not a calibrated probability."""
    p = b["model"].predict_proba(prepare_features(df, b))[:, 1]
    return p, p >= b["threshold"]
'''
exec(INFERENCE_PY)                                   # defines prepare_features / score_orders here too

TOP_CATS = TR["category"].value_counts().head(20).index.tolist() if "category" in FEATS else []
tmp = TR[FEATS].copy()
if "category" in FEATS:
    tmp["category"] = tmp["category"].where(tmp["category"].isin(TOP_CATS), "other")
CODES = {c: {v: i for i, v in enumerate(sorted(tmp[c].astype(str).unique()))} for c in CAT}
if "category" in CODES and "other" not in CODES["category"]:
    CODES["category"]["other"] = len(CODES["category"])
B = dict(features=FEATS, categorical=CAT, codes=CODES, top_categories=TOP_CATS)
print({c: len(v) for c, v in CODES.items()}, "values per categorical column")
display(prepare_features(TR.head(3), B))

In [ ]:
# The three candidate model types (all handle the imbalance with class weights, no rows are invented)
def make_model(name):
    if name == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=6, min_samples_leaf=40, l2_regularization=1.0,
                                              categorical_features=[c in CAT for c in FEATS], class_weight="balanced", random_state=SEED)
    if name == "random_forest":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("rf", RandomForestClassifier(n_estimators=300, min_samples_leaf=20, max_features="sqrt",
                                                       class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))])
    if name == "logistic_regression":
        pre = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
                                 ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
        return Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=2000, C=0.5, class_weight="balanced"))])
    raise ValueError(name)


def eval_scores(y, s, k=CFG["TOP_K"]):
    y, s = np.asarray(y), np.asarray(s)
    kk = max(1, int(len(y) * k)); top = np.argsort(-s)[:kk]
    return dict(roc_auc=roc_auc_score(y, s), pr_auc=average_precision_score(y, s),
                recall_at_topk=y[top].sum() / max(y.sum(), 1), base_rate=y.mean())


# Model-type choice: forward-chaining CV inside 2017 (train on the past, test on the next two months, 30-day embargo)
cv_src = clean[clean.split == "train"].sort_values("purchase_ts").reset_index(drop=True)
Xcv, ycv = prepare_features(cv_src, B), cv_src.is_late.values
folds = [("2017-07-01", "2017-09-01"), ("2017-09-01", "2017-11-01"), ("2017-11-01", "2018-01-01")]
rows = []
for name in CFG["CV_MODELS"]:
    for lo, hi in folds:
        lo, hi = pd.Timestamp(lo), pd.Timestamp(hi)
        tri = (cv_src.purchase_ts < lo - pd.Timedelta(days=30)).values
        tei = ((cv_src.purchase_ts >= lo) & (cv_src.purchase_ts < hi)).values
        m = make_model(name); m.fit(Xcv[tri], ycv[tri])
        rows.append(dict(model=name, test_months=f"{lo:%Y-%m}..{(hi - pd.Timedelta(days=1)):%Y-%m}", n_train=int(tri.sum()),
                         n_test=int(tei.sum()), late_in_test=int(ycv[tei].sum()), **eval_scores(ycv[tei], m.predict_proba(Xcv[tei])[:, 1])))
cv = pd.DataFrame(rows)
display(cv)
cv_summary = cv.groupby("model")[["roc_auc", "pr_auc", "recall_at_topk"]].mean().sort_values("pr_auc", ascending=False)
display(cv_summary)
BEST = CFG["FORCE_MODEL"] or cv_summary.index[0]
print(f"\nSelected model type: {BEST}   (highest average PR-AUC across the 3 forward-chaining folds; 3 folds is a small sample, so close results are ties)")
fig, ax = plt.subplots(figsize=(8, 3.6))
cv_summary.pr_auc.sort_values().plot.barh(ax=ax, color=C_OK); ax.set_title("Average PR-AUC across forward-chaining folds (2017)"); ax.set_xlabel("PR-AUC")
plt.tight_layout(); plt.show()

In [ ]:
# Final fit on the training set from the data plan
TRfit = TR
if CFG["ONTIME_PER_LATE"]:
    late_rows = TR[TR.is_late == 1]
    on_rows = TR[TR.is_late == 0].sample(n=min(len(late_rows) * CFG["ONTIME_PER_LATE"], int((TR.is_late == 0).sum())), random_state=SEED)
    TRfit = pd.concat([late_rows, on_rows]).sample(frac=1, random_state=SEED)
Xtr, ytr = prepare_features(TRfit, B), TRfit.is_late.values
Xva, yva = prepare_features(VA, B), VA.is_late.values
Xte, yte = prepare_features(TE, B), TE.is_late.values
print(f"fitting {BEST} on {len(TRfit):,} rows: {int(ytr.sum()):,} late + {int((ytr == 0).sum()):,} on-time ({ytr.mean() * 100:.1f}% late)")
t0 = time.time()
model = make_model(BEST).fit(Xtr, ytr)
print(f"done in {time.time() - t0:.0f} s")

In [ ]:
# Validation: set the alert threshold from the on-time orders (false-alarm budget)
p_va = model.predict_proba(Xva)[:, 1]
p_tr = model.predict_proba(Xtr)[:, 1]
on_va = p_va[yva == 0]
thr = float(np.quantile(on_va, 1 - CFG["FPR_TARGET"]))
fpr_tbl = pd.DataFrame({"share of on-time orders flagged (false alarms)": [0.05, 0.10, 0.15, 0.20, 0.30]})
fpr_tbl["score threshold"] = [np.quantile(on_va, 1 - f) for f in fpr_tbl.iloc[:, 0]]
display(fpr_tbl)
print(f"chosen alert threshold {thr:.3f}  (flags {CFG['FPR_TARGET'] * 100:.0f}% of validation on-time orders)")
print(f"overfitting check: median score of on-time TRAINING orders {np.median(p_tr[ytr == 0]):.3f} vs on-time VALIDATION orders {np.median(on_va):.3f}")
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(on_va, bins=50, color=C_OK, alpha=0.85); ax.axvline(thr, c=C_LATE, ls="--", label=f"threshold {thr:.2f}")
ax.set_title(f"Scores of the {len(on_va):,} on-time validation orders (every alert here is a false alarm)"); ax.set_xlabel("risk score"); ax.legend()
plt.tight_layout(); plt.show()
if yva.sum() > 0:                                       # only when validation contains late orders (USE_VALID_LATE_IN_TRAIN = False)
    print("validation metrics:", {k: round(v, 3) for k, v in eval_scores(yva, p_va).items()})

In [ ]:
# Final test: real May-Aug 2018 orders (used once)
p_te = model.predict_proba(Xte)[:, 1]
res_te = eval_scores(yte, p_te)
flag = p_te >= thr
tn, fp, fn, tp = confusion_matrix(yte, flag).ravel()
# simple baseline: late rate of the customer's state in the training data
st = TRfit.groupby("customer_state").is_late.agg(["sum", "count"]); base_rate_tr = TRfit.is_late.mean()
st_rate = (st["sum"] + 5 * base_rate_tr) / (st["count"] + 5)
p_base = TE.customer_state.map(st_rate).fillna(base_rate_tr).values
res_base = eval_scores(yte, p_base)
out = pd.DataFrame({"Model 1": res_te, "state-only baseline": res_base, "random guess": dict(roc_auc=0.5, pr_auc=yte.mean(), recall_at_topk=CFG["TOP_K"], base_rate=yte.mean())}).T
display(out.round(3))
print(f"\nAt the validation threshold {thr:.3f}: alerts {flag.mean() * 100:.1f}% of orders | caught {tp:,} of {tp + fn:,} late orders "
      f"(recall {tp / (tp + fn) * 100:.1f}%) | precision {tp / max(tp + fp, 1) * 100:.1f}% | false alarms {fp / (fp + tn) * 100:.1f}% of on-time orders")
print(pd.DataFrame([[tn, fp], [fn, tp]], index=["actually on time", "actually late"], columns=["not flagged", "flagged"]))

fig, axs = plt.subplots(2, 2, figsize=(13, 9))
fpr_, tpr_, _ = roc_curve(yte, p_te); axs[0, 0].plot(fpr_, tpr_, c=C_LATE, lw=2, label=f"Model 1 (AUC {res_te['roc_auc']:.3f})")
f2, t2, _ = roc_curve(yte, p_base); axs[0, 0].plot(f2, t2, c=C_GREY, label=f"state baseline ({res_base['roc_auc']:.3f})"); axs[0, 0].plot([0, 1], [0, 1], "k:", lw=1)
axs[0, 0].set_title("ROC curve (test)"); axs[0, 0].set_xlabel("false alarm rate"); axs[0, 0].set_ylabel("late orders caught"); axs[0, 0].legend()
pr_, rc_, _ = precision_recall_curve(yte, p_te); axs[0, 1].plot(rc_, pr_, c=C_LATE, lw=2, label=f"Model 1 (PR-AUC {res_te['pr_auc']:.3f})")
axs[0, 1].axhline(yte.mean(), c=C_GREY, ls="--", label=f"random ({yte.mean():.3f})"); axs[0, 1].set_title("Precision-recall curve (test)"); axs[0, 1].set_xlabel("recall"); axs[0, 1].set_ylabel("precision"); axs[0, 1].legend()
o = np.argsort(-p_te); cum = np.cumsum(yte[o]) / yte.sum(); x_ = np.arange(1, len(o) + 1) / len(o)
axs[1, 0].plot(x_ * 100, cum * 100, c=C_LATE, lw=2, label="Model 1"); axs[1, 0].plot([0, 100], [0, 100], "k:", label="random")
axs[1, 0].set_title("Gains: late orders caught vs share of orders alerted"); axs[1, 0].set_xlabel("% of orders alerted (riskiest first)"); axs[1, 0].set_ylabel("% of late orders caught"); axs[1, 0].legend()
axs[1, 1].hist(p_te[yte == 0], bins=50, alpha=0.7, color=C_OK, density=True, label="on time"); axs[1, 1].hist(p_te[yte == 1], bins=50, alpha=0.7, color=C_LATE, density=True, label="late")
axs[1, 1].axvline(thr, c="k", ls="--", label="alert threshold"); axs[1, 1].set_title("Score distribution on test"); axs[1, 1].set_xlabel("risk score"); axs[1, 1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Test results month by month (the late rate and the model both move with the surge)
mt = TE.assign(score=p_te, flag=flag, month=TE.purchase_ts.dt.to_period("M").astype(str))
rows = []
for mth, g in mt.groupby("month"):
    if g.is_late.nunique() < 2:
        continue
    r = eval_scores(g.is_late.values, g.score.values)
    rows.append(dict(month=mth, orders=len(g), late=int(g.is_late.sum()), late_pct=g.is_late.mean() * 100, roc_auc=r["roc_auc"], pr_auc=r["pr_auc"],
                     recall_at_topk=r["recall_at_topk"], alerted_pct=g.flag.mean() * 100, recall_at_threshold=g[g.is_late == 1].flag.mean() * 100))
monthly = pd.DataFrame(rows); display(monthly.round(3))

In [ ]:
# Which columns does Model 1 rely on? (permutation importance on a test sample; descriptive, after the final evaluation)
sub = TE.sample(n=min(10000, len(TE)), random_state=SEED)
pi = permutation_importance(model, prepare_features(sub, B), sub.is_late.values, scoring="average_precision", n_repeats=3, random_state=SEED, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values()
fig, ax = plt.subplots(figsize=(9, 6)); imp.plot.barh(ax=ax, color=C_OK)
ax.set_title("Permutation importance (drop in PR-AUC when a column is shuffled)"); ax.set_xlabel("importance")
plt.tight_layout(); plt.show()

In [ ]:
# Save everything the Kafka scorer needs: model, encoders, threshold, and the exact feature-preparation code
bundle = dict(model=model, model_name=BEST, features=FEATS, categorical=CAT, codes=CODES, top_categories=TOP_CATS, threshold=thr,
              fpr_target=CFG["FPR_TARGET"], config=CFG, sklearn_version=sklearn.__version__,
              trained_on=dict(rows=int(len(TRfit)), late=int(ytr.sum()), first=str(TRfit.purchase_ts.min().date()), last=str(TRfit.purchase_ts.max().date())),
              test_metrics={k: float(v) for k, v in res_te.items()})
joblib.dump(bundle, "model1_bundle.joblib")
open("model1_inference.py", "w").write(INFERENCE_PY)
json.dump(dict(model=BEST, threshold=thr, test=res_te, baseline=res_base, cv=cv_summary.to_dict(), data_plan=summ.to_dict()), open("model1_metrics.json", "w"), indent=2, default=float)

# check that the saved bundle scores raw rows the way the Kafka scorer will
chk = joblib.load("model1_bundle.joblib")
p_chk, f_chk = score_orders(TE.head(5), chk)
print("saved: model1_bundle.joblib, model1_inference.py, model1_metrics.json")
print("example scoring of 5 test orders:", np.round(p_chk, 3), f_chk)
try:
    from google.colab import files
    for f_ in ["model1_bundle.joblib", "model1_inference.py", "model1_metrics.json"]:
        files.download(f_)
except Exception:
    print("(not running in Colab: files are in the current folder)")

### What to take from this section
* **Model 1 table** (test): compare Model 1 with the state-only baseline and the random guess. Expect a moderate improvement, not a spectacular one: purchase-time information is limited, as the EDA showed.
* **Alert threshold**: set from on-time validation orders, so the false-alarm budget is explicit. The test confusion matrix shows what that budget buys in late orders caught.
* **Model choice**: taken from forward-chaining CV on 2017, not from validation or test.
* **Next**: Model 3 (handoff stage) uses the same data plan with the extra columns `approval_lag_h`, `handling_days`, `seller_overrun_days`, `post_approval_days`.

## 14. Choosing the best 15 of the 23 Model 1 columns
*Question: which of the 23 checkout-time columns actually help, so we can keep only the useful ~15?*

**Run this section in Colab after uploading the NEW `orders_clean.csv` (39 columns) and `feature_registry.json`.** It is self-contained. Nothing here has been run for you.

| Step | What it does | In simple words |
|---|---|---|
| 1 | Single-column test | Does this column separate late from on-time orders at all? |
| 2 | Duplicate check | Which columns say the same thing? (keep the better one) |
| 3 | Leave-one-out | Retrain without one column: how much does the score drop? |
| 4 | Backward elimination | Remove the weakest column, repeat, and watch the score as columns disappear |
| 5 | Final check | Train with all 23 and with the chosen 15; compare once on the test set |

**Rules that keep it honest:** steps 1 to 4 use **only the 2017 training period**, in time order (train on the past, check on the next two months, 30-day gap). The validation and test sets are not used to choose columns. The test set is touched once, in step 5.
Expect roughly 5 to 15 minutes in Colab (many small model fits).

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, joblib
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12, "figure.figsize": (10, 4.5),
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
C_LATE, C_OK, C_GREY, C_ACC = "#d1495b", "#2e86ab", "#8d99ae", "#edae49"

CFG14 = dict(
    CLEAN_FILE="orders_clean.csv", REGISTRY_FILE="feature_registry.json",
    FINAL_K=15,                       # how many columns to keep
    MIN_K=8,                          # how far backward elimination goes
    SEED=42, TOP_K=0.10, FPR_TARGET=0.10,
    USE_VALID_LATE_IN_TRAIN=True,     # same data plan as section 13
)
SEED = CFG14["SEED"]


def find_data_file(name, dirs=(".", "/content", "/content/drive/MyDrive", "data", "/mnt/user-data/uploads", "/mnt/user-data/outputs")):
    for d in dirs:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found. Upload it in the Colab file panel (left sidebar) or mount Google Drive.")


def auc_uni(x, y):
    x = pd.Series(np.asarray(x, dtype=float)).reset_index(drop=True); y = np.asarray(y); ok = x.notna().values
    if ok.sum() < 500 or y[ok].sum() == 0:
        return np.nan
    r = rankdata(x[ok]); yy = y[ok]; n1 = yy.sum(); n0 = len(yy) - n1
    return (r[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def eval_scores(y, s, k=CFG14["TOP_K"]):
    y, s = np.asarray(y), np.asarray(s); kk = max(1, int(len(y) * k)); top = np.argsort(-s)[:kk]
    caught = y[top].sum()
    return dict(roc_auc=roc_auc_score(y, s), pr_auc=average_precision_score(y, s), base_rate=y.mean(),
                recall_top10=caught / max(y.sum(), 1), precision_top10=caught / kk)


clean = pd.read_csv(find_data_file(CFG14["CLEAN_FILE"]), parse_dates=["purchase_ts"])
NEW4 = ["route_recent_days", "promise_buffer_route", "promise_buffer_state", "order_load_ratio"]
FALLBACK = ["promise_days", "purchase_hour", "purchase_weekday", "price", "freight", "n_items", "weight_g", "volume_cm3", "category",
            "customer_state", "seller_state", "same_state", "dist_km", "pay_type", "installments", "seller_prior_orders",
            "seller_late_rate_prior", "seller_overrun_rate_prior", "seller_handling_days_prior"] + NEW4
try:
    reg = json.load(open(find_data_file(CFG14["REGISTRY_FILE"])))
    ALL = reg["models"]["model_1"]["features"]; CAT_ALL = [c for c in reg["categorical_features"] if c in ALL]
except FileNotFoundError:
    ALL, CAT_ALL = FALLBACK, ["category", "customer_state", "seller_state", "pay_type"]
NUM_ALL = [c for c in ALL if c not in CAT_ALL]
missing = [c for c in ALL if c not in clean.columns]
assert not missing, f"Your orders_clean.csv is the OLD version. Missing columns: {missing}. Upload the new file (39 columns)."
print(f"{len(ALL)} candidate columns ({len(NUM_ALL)} numeric, {len(CAT_ALL)} categorical); scikit-learn {sklearn.__version__}")

# same feature preparation code as section 13 / the Kafka scorer
INFERENCE_PY = '''
import numpy as np, pandas as pd, joblib


def load_bundle(path="model1_bundle.joblib"):
    return joblib.load(path)


def prepare_features(df, b):
    """df must contain the Model 1 columns (same names as orders_stream.csv). Returns the model matrix."""
    X = df[b["features"]].copy()
    if "category" in X.columns:
        X["category"] = X["category"].where(X["category"].isin(b["top_categories"]), "other")
    for c in b["categorical"]:
        X[c] = X[c].astype(str).map(b["codes"][c]).fillna(-1).astype(int)     # unseen value -> -1 (treated as missing)
    return X


def score_orders(df, b):
    """Returns (risk score, alert flag). Score is a ranking score, not a calibrated probability."""
    p = b["model"].predict_proba(prepare_features(df, b))[:, 1]
    return p, p >= b["threshold"]
'''
exec(INFERENCE_PY)


def build_B(df, feats):
    cat = [c for c in CAT_ALL if c in feats]
    top = df["category"].value_counts().head(20).index.tolist() if "category" in feats else []
    tmp = df[list(feats)].copy()
    if "category" in feats:
        tmp["category"] = tmp["category"].where(tmp["category"].isin(top), "other")
    codes = {c: {v: i for i, v in enumerate(sorted(tmp[c].astype(str).unique()))} for c in cat}
    if "category" in codes and "other" not in codes["category"]:
        codes["category"]["other"] = len(codes["category"])
    return dict(features=list(feats), categorical=cat, codes=codes, top_categories=top)


def make_hgb(feats):
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=5, min_samples_leaf=50, l2_regularization=1.0,
                                          categorical_features=[f in CAT_ALL for f in feats], class_weight="balanced", random_state=SEED)


# 2017 only, in time order: train on the past, test on the next two months, 30-day gap
cv_src = clean[clean.split == "train"].sort_values("purchase_ts").reset_index(drop=True)
ycv = cv_src.is_late.values
X_ALL = prepare_features(cv_src, build_B(cv_src, ALL))
FOLD_IDX = []
for lo, hi in [("2017-07-01", "2017-09-01"), ("2017-09-01", "2017-11-01"), ("2017-11-01", "2018-01-01")]:
    lo, hi = pd.Timestamp(lo), pd.Timestamp(hi)
    FOLD_IDX.append(((cv_src.purchase_ts < lo - pd.Timedelta(days=30)).values, ((cv_src.purchase_ts >= lo) & (cv_src.purchase_ts < hi)).values))
print("selection data (2017 only):", len(cv_src), "orders,", int(ycv.sum()), "late;  folds:", [(int(a.sum()), int(b.sum())) for a, b in FOLD_IDX])


def cv_score(feats):
    prs, rocs = [], []
    for tri, tei in FOLD_IDX:
        m = make_hgb(feats).fit(X_ALL.loc[tri, feats], ycv[tri])
        p = m.predict_proba(X_ALL.loc[tei, feats])[:, 1]
        prs.append(average_precision_score(ycv[tei], p)); rocs.append(roc_auc_score(ycv[tei], p))
    return dict(pr_auc=float(np.mean(prs)), roc_auc=float(np.mean(rocs)))

In [ ]:
# STEP 1: each column on its own (second half of 2017; category rates are learned from the first half only)
early = (cv_src.purchase_ts < "2017-07-01").values
rows = []
for f in ALL:
    if f in CAT_ALL:
        g = cv_src[early].groupby(f).is_late.agg(["sum", "count"]); r_ = (g["sum"] + 5 * ycv[early].mean()) / (g["count"] + 5)
        s_ = cv_src.loc[~early, f].map(r_).fillna(ycv[early].mean())
    else:
        s_ = cv_src.loc[~early, f]
    a_ = auc_uni(s_, ycv[~early])
    rows.append(dict(feature=f, kind="category" if f in CAT_ALL else "number", auc=a_, strength=max(a_, 1 - a_), filled_pct=cv_src[f].notna().mean() * 100))
uni = pd.DataFrame(rows).sort_values("strength", ascending=False).reset_index(drop=True)
display(uni)
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(uni.feature[::-1], uni.strength[::-1] - 0.5, left=0.5, color=[C_LATE if f in NEW4 else C_OK for f in uni.feature[::-1]])
ax.axvline(0.5, c="black", lw=1); ax.set_xlim(0.47, max(0.7, uni.strength.max() + 0.02))
ax.set_title("STEP 1: single-column strength (AUC, 0.5 = no signal). Red = the 4 new columns"); ax.set_xlabel("AUC (direction ignored)")
plt.tight_layout(); plt.show()

In [ ]:
# STEP 2: columns that say the same thing (Spearman correlation of 0.8 or more)
cm = cv_src[NUM_ALL].corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt=".2f", annot_kws={"size": 7}, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.3)
ax.set_title("STEP 2: correlation between numeric columns"); plt.tight_layout(); plt.show()
strength = dict(zip(uni.feature, uni.strength))
pairs = [(a, b, cm.loc[a, b]) for i, a in enumerate(NUM_ALL) for b in NUM_ALL[i + 1:] if abs(cm.loc[a, b]) >= 0.8]
dup = pd.DataFrame([dict(column_a=a, column_b=b, correlation=r, suggest_dropping=(a if strength[a] < strength[b] else b)) for a, b, r in pairs])
print("pairs that overlap strongly:" if len(dup) else "no pair of numeric columns correlates at 0.8 or more")
if len(dup):
    display(dup)

In [ ]:
# STEP 3: leave-one-out. Retrain without one column and see how much the score drops (positive loss = the column helps)
t0 = time.time()
base = cv_score(ALL)
rows = []
for f in ALL:
    r_ = cv_score([x for x in ALL if x != f])
    rows.append(dict(feature=f, pr_auc_loss=base["pr_auc"] - r_["pr_auc"], roc_auc_loss=base["roc_auc"] - r_["roc_auc"]))
loo = pd.DataFrame(rows).sort_values("pr_auc_loss", ascending=False).reset_index(drop=True)
print(f"all {len(ALL)} columns: PR-AUC {base['pr_auc']:.4f}, ROC-AUC {base['roc_auc']:.4f}   ({time.time() - t0:.0f} s)")
display(loo)
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(loo.feature[::-1], loo.pr_auc_loss[::-1], color=[C_OK if v > 0 else C_GREY for v in loo.pr_auc_loss[::-1]])
ax.axvline(0, c="black", lw=1); ax.set_title("STEP 3: PR-AUC lost when the column is removed (negative = score improves without it)"); ax.set_xlabel("PR-AUC loss")
plt.tight_layout(); plt.show()

In [ ]:
# STEP 4: backward elimination. Remove the weakest column (by permutation importance on the held-out months), repeat.
def rfe_path(feats, min_k):
    feats = list(feats); path = []
    while True:
        imps = pd.Series(0.0, index=feats); prs, rocs = [], []
        for tri, tei in FOLD_IDX:
            m = make_hgb(feats).fit(X_ALL.loc[tri, feats], ycv[tri])
            Xt, yt = X_ALL.loc[tei, feats], ycv[tei]
            p = m.predict_proba(Xt)[:, 1]; prs.append(average_precision_score(yt, p)); rocs.append(roc_auc_score(yt, p))
            pi = permutation_importance(m, Xt, yt, scoring="average_precision", n_repeats=3, random_state=SEED, n_jobs=1)
            imps += pd.Series(pi.importances_mean, index=feats) / len(FOLD_IDX)
        path.append(dict(n_features=len(feats), pr_auc=float(np.mean(prs)), roc_auc=float(np.mean(rocs)), features=list(feats), dropped_next=None))
        if len(feats) <= min_k:
            break
        weakest = imps.idxmin(); path[-1]["dropped_next"] = weakest; feats.remove(weakest)
    return path


t0 = time.time()
path = rfe_path(ALL, CFG14["MIN_K"])
pt = pd.DataFrame(path)[["n_features", "pr_auc", "roc_auc", "dropped_next"]]
display(pt)
best_pr = pt.pr_auc.max()
smallest_ok = int(pt[pt.pr_auc >= 0.98 * best_pr].n_features.min())
K = CFG14["FINAL_K"]
FINAL = next(p["features"] for p in path if p["n_features"] == K)
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(pt.n_features, pt.pr_auc, marker="o", c=C_LATE, label="PR-AUC"); ax.set_xlabel("number of columns kept"); ax.set_ylabel("PR-AUC", color=C_LATE)
ax2 = ax.twinx(); ax2.plot(pt.n_features, pt.roc_auc, marker="s", c=C_OK, label="ROC-AUC"); ax2.set_ylabel("ROC-AUC", color=C_OK); ax2.grid(False)
ax.axvline(K, c="black", ls="--"); ax.invert_xaxis(); ax.set_title(f"STEP 4: score as columns are removed (dashed line = the {K} we keep)")
plt.tight_layout(); plt.show()
print(f"{time.time() - t0:.0f} s | best PR-AUC {best_pr:.4f}; the smallest set within 2% of the best has {smallest_ok} columns")
sel = pd.DataFrame({"feature": FINAL}).merge(uni[["feature", "strength"]], on="feature", how="left").merge(loo[["feature", "pr_auc_loss"]], on="feature", how="left")
print(f"\nTOP {K} COLUMNS FOR MODEL 1:"); display(sel)
print("dropped:", [f for f in ALL if f not in FINAL])

In [ ]:
# STEP 5: final check. Train with all 23 and with the chosen set on the data plan's training rows; test once.
tr_main = clean[clean.split == "train"]
if CFG14["USE_VALID_LATE_IN_TRAIN"]:
    TR = pd.concat([tr_main, clean[(clean.split == "valid") & (clean.is_late == 1)]]); VA = clean[(clean.split == "valid") & (clean.is_late == 0)]
else:
    TR, VA = tr_main, clean[clean.split == "valid"]
TE = clean[clean.split == "test"]
B_FIN = build_B(TR, ALL)
Xtr, Xva, Xte = prepare_features(TR, B_FIN), prepare_features(VA, B_FIN), prepare_features(TE, B_FIN)
ytr, yva, yte = TR.is_late.values, VA.is_late.values, TE.is_late.values
print(f"training {len(TR):,} rows ({int(ytr.sum()):,} late) | validation {len(VA):,} ({int(yva.sum())} late) | test {len(TE):,} ({int(yte.sum()):,} late)")
out, fitted = {}, {}
for name, feats in [(f"all {len(ALL)} columns", ALL), (f"top {K} columns", FINAL)]:
    m = make_hgb(feats).fit(Xtr[feats], ytr); fitted[name] = (m, feats)
    p_te = m.predict_proba(Xte[feats])[:, 1]; on_va = m.predict_proba(Xva[feats])[:, 1][yva == 0]
    thr = float(np.quantile(on_va, 1 - CFG14["FPR_TARGET"])); flag = p_te >= thr
    r = eval_scores(yte, p_te)
    r.update(lift_top10=r["precision_top10"] / r["base_rate"], threshold=thr, alert_rate=flag.mean(),
             false_alarm_rate=flag[yte == 0].mean(), recall_at_threshold=flag[yte == 1].mean())
    out[name] = r
res14 = pd.DataFrame(out).T
display(res14.round(3))
print(f"\nThe validation threshold was set to flag {CFG14['FPR_TARGET'] * 100:.0f}% of on-time orders; on test it flagged "
      + ", ".join(f"{v * 100:.1f}% ({k})" for k, v in res14.false_alarm_rate.items()) + ". A gap means validation and test behave differently (drift).")

In [ ]:
# Save the chosen columns and the final model for the Kafka scorer
name_fin = f"top {K} columns"; m_fin, f_fin = fitted[name_fin]
B_SEL = build_B(TR, f_fin)
bundle = dict(model=m_fin, model_name="hist_gradient_boosting", features=f_fin, categorical=B_SEL["categorical"], codes=B_SEL["codes"],
              top_categories=B_SEL["top_categories"], threshold=float(res14.loc[name_fin, "threshold"]), fpr_target=CFG14["FPR_TARGET"],
              sklearn_version=sklearn.__version__, test_metrics={k: float(v) for k, v in res14.loc[name_fin].items()})
joblib.dump(bundle, "model1_bundle_top15.joblib")
open("model1_inference.py", "w").write(INFERENCE_PY)
json.dump(dict(selected_features=f_fin, dropped=[f for f in ALL if f not in f_fin], elimination_path=[{k: v for k, v in p.items() if k != "features"} for p in path],
               single_column=uni.to_dict("records"), leave_one_out=loo.to_dict("records"), final_comparison=res14.to_dict("index")),
          open("model1_selected_features.json", "w"), indent=2, default=float)
chk = joblib.load("model1_bundle_top15.joblib"); p_chk, f_chk = score_orders(TE.head(5), chk)
print("saved: model1_bundle_top15.joblib, model1_inference.py, model1_selected_features.json")
print("example scoring of 5 test orders:", np.round(p_chk, 3), f_chk)
try:
    from google.colab import files
    for f_ in ["model1_bundle_top15.joblib", "model1_inference.py", "model1_selected_features.json"]:
        files.download(f_)
except Exception:
    print("(not running in Colab: files are in the current folder)")

### How to read the results
* **Step 1** shows which columns carry any signal on their own. The 4 new columns are in red.
* **Step 3** shows which columns the model really needs. A column with zero or negative loss can go.
* **Step 4** shows the score as columns are removed. If it stays flat from 23 down to 15 (or lower), the dropped columns were not helping, which is the evidence for the final list.
* **Step 5** compares all 23 columns with the chosen 15 on the test set. Similar scores with fewer columns is the result we want.
* The saved `model1_selected_features.json` is the list to put into `feature_registry.json` for Model 1.